In [ ]:
# Standard library
import os
import glob
import re
import csv
import gzip
import math
import time
import timeit
import random
import copy
from datetime import date

# NumPy, SciPy, and Pandas
import numpy as np
import pandas as pd
from scipy.stats import binom

# Matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import (
    MultipleLocator, FormatStrFormatter, AutoMinorLocator, LinearLocator
)

# BioPython
from Bio import Entrez, SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Align import MultipleSeqAlignment

# Other bioinformatics tools
import pysam
from pyfaidx import Fasta

# !pip install XlsxWriter
# !pip install networkx
import networkx as nx
from itertools import combinations

import os
import pandas as pd
from pathlib import Path

In [ ]:
# Lists of colors for plots
c0 = (0.76, 0.76, 0.76)
c1 = (1.00, 0.18, 0.33);
c2 = (1.00, 0.23, 0.19);
c3 = (1.00, 0.58, 0.00);
c4 = (1.00, 0.80, 0.00);
c5 = (0.30, 0.85, 0.39);
c6 = (0.35, 0.78, 0.98);
c7 = (0.20, 0.67, 0.86);
c8 = (0.00, 0.48, 1.00);
c9 = (0.35, 0.34, 0.84);
c10 = (0.00, 0.31, 0.57);
c11 = (0.12, 0.29, 0.69);
c12 = (0.17, 0.17, 0.42);
c13 = (1.00, 1.00, 1.00);
c14 = (0.77, 0.04, 0.00);

#define the colors from colorbrewer2
orange1 = '#feedde'
orange2 = '#fdbe85'
orange3 = '#fd8d3c'
orange4 = '#e6550d'
orange5 = '#a63603'
blue1 = '#eff3ff'
blue2 = '#bdd7e7'
blue3 = '#6baed6'
blue4 = '#3182bd'
blue5 = '#08519c'
green1 = '#edf8e9'
green2 = '#bae4b3'
green3 = '#74c476'
green4 = '#31a354'
green5 = '#006d2c'
grey1 = '#f7f7f7'
grey2 = '#cccccc'
grey3 = '#969696'
grey4 = '#636363'
grey5 = '#252525'
purple1 = '#f2f0f7'
purple2 = '#cbc9e2'
purple3 = '#9e9ac8'
purple4 = '#756bb1'
purple5 = '#54278f'
red1 = '#fee5d9'
red2 = '#fcae91'
red3 = '#fb6a4a'
red4 = '#de2d26'
red5 = '#a50f15'


In [ ]:
def extract_position(position_info):
    coordinate = position_info.split(' ')[2]
    coordinate_number = coordinate.strip('><')
    return int(coordinate_number)

In [ ]:
def extract_position2(position_info):
    try:
        parts = position_info.split(' ')
        if len(parts) < 3:
            return None
        coordinate = parts[2].strip('><')
        return int(coordinate)
    except (ValueError, AttributeError, IndexError):
        return None

In [ ]:
def maximum_of_two_columns(depth1, depth2):
    num1 = pd.to_numeric(depth1, errors='coerce')
    num2 = pd.to_numeric(depth2, errors='coerce')
    maxdepth = np.fmax(num1.fillna(-np.inf), num2.fillna(-np.inf))
    maxdepth = maxdepth.replace(-np.inf, np.nan)
    return maxdepth

In [ ]:
def assign_proximity_groups(subdf): #for grouping together translocations whose coordinates are within 500bp of each other
    positions = subdf['position'].values
    subgroup = [0]
    current_group = 0
    for i in range(1, len(positions)):
        if positions[i] - positions[i - 1] > 500:
            current_group += 1
        subgroup.append(current_group)
    subdf['subgroup'] = current_group_base + np.array(subgroup)
    return subdf

In [ ]:
def extract_chromosome(value):
    parts = value.split(' ')
    return parts[1]

In [ ]:
def fusion_genes(left_gene, right_gene):
    return left_gene.astype(str)+'::'+right_gene.astype(str)

In [ ]:
def filter_and_group_chromosomal_rearrangements(sample_name):
    df = pd.read_csv(sample_name+'/'+sample_name+'_translocations_found_just_those_specifically_targeted_both_sides_panel.csv')
    df['LEFT COORDINATE'] = df["LEFT SIDE (5') OF NON-INVERTED BREAKPOINT"].apply(extract_position)
    df['RIGHT COORDINATE'] = df["RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT"].apply(extract_position)
    df = df[df['LEFT GENE'] != df['RIGHT GENE']] #remove rows where the rearrangement supposedly occurs within the same gene
    df['max_normal_depth'] = maximum_of_two_columns(df['normal_depth_left'], df['normal_depth_right'])
    df['max_VAF'] = maximum_of_two_columns(df['VAF_left'], df['VAF_right'])
    
    # STEP 1: Reset and label rows
    df = df.reset_index(drop=True)  # start fresh
    df['row_id'] = df.index         # assign a unique integer row ID
    
    df['GENE_PAIR'] = fusion_genes(df['LEFT GENE'], df['RIGHT GENE'])
    
    # STEP 2: Sort
    df = df.sort_values(by=['NOTATION', 'GENE_PAIR', 'LEFT GENE', 'RIGHT GENE', 'TYPE', 'LEFT COORDINATE', 'RIGHT COORDINATE'])
    
    # STEP 3: Build groups
    grouped_blocks = []
    group_id_base = 0
    
    for _, group in df.groupby(['NOTATION', 'LEFT GENE', 'RIGHT GENE', 'TYPE']):
        g = group.copy()
        G = nx.Graph()    
        row_ids = g['row_id'].tolist()
        G.add_nodes_from(row_ids)
        g['subgroup'] = np.nan
    
        for i, j in combinations(g.index, 2):
            row_i = g.loc[i]
            row_j = g.loc[j]
            if (
                abs(row_i['LEFT COORDINATE'] - row_j['LEFT COORDINATE']) <= 500
                and abs(row_i['RIGHT COORDINATE'] - row_j['RIGHT COORDINATE']) <= 500
            ):
                G.add_edge(row_i['row_id'], row_j['row_id'])
    
        for group_num, component in enumerate(nx.connected_components(G), start=group_id_base):
            idx = g['row_id'].isin(component)
            g.loc[idx, 'subgroup'] = group_num
    
        group_id_base = int(g['subgroup'].max()) + 1
        grouped_blocks.append(g)
    
    # STEP 4: Combine and annotate group sizes
    df_with_groups = pd.concat(grouped_blocks).reset_index(drop=True)
    df_with_groups['subgroup'] = df_with_groups['subgroup'].astype(int)
    group_sizes = df_with_groups['subgroup'].value_counts()
    df_with_groups['subgroup_size'] = df_with_groups['subgroup'].map(group_sizes)
    
    # STEP 5: Rank gene pairs by frequency
    gene_pair_sizes = df_with_groups['GENE_PAIR'].value_counts()
    gene_pair_rank = {gp: i for i, gp in enumerate(gene_pair_sizes.index, start=1)}
    df_with_groups['SORT_ORDER'] = df_with_groups['GENE_PAIR'].map(gene_pair_rank)
    
    summary_cols = [
        'D_NO_OVERLAP',
        ('D_NPP', 'C_SUPP_NPP'),
        ('D_NPP',),
        ('D_NPP', 'softclip_mapping'),
        ('D_SUPP_PP', 'C_PP'),
        ('D_SUPP_NPP', 'D_NPP'),
        ('concordant_1_end_mapping',),
        ('D_NPP_MPP', 'softclip_mapping'),
        ('C_SUPP_NPP', 'softclip_mapping'),
        ('D_SUPP_NPP', 'D_NPP_MPP'),
        ('C_PP',),
        ('D_SUPP_NPP', 'D_NPP', 'C_SUPP_NPP'),
        ('D_SUPP_NPP', 'C_NPP'),
        ('D_SUPP_NPP', 'C_SUPP_NPP'),
        ('D_SUPP_NPP', 'softclip_mapping'),
        ('D_NPP_MPP', 'C_SUPP_NPP'),
        ('C_PP', 'softclip_mapping'),
        ('C_NPP', 'softclip_mapping'),
        ('D_NPP_LEFT_INV', 'C_SUPP_NPP_LEFT'),
        ('D_NPP_LEFT_INV',),
        ('D_NPP_LEFT_INV', 'inv_softclip_mapping'),
        ('D_SUPP_NPP_LEFT', 'D_NPP_LEFT_INV'),
        ('D_SUPP_PP_INV', 'C_PP'),
        'D_NO_OVERLAP_INV',
        ('D_SUPP_NPP_INV', 'D_NPP'),
        ('D_SUPP_NPP_INV', 'D_NPP_LEFT_INV'),
        ('D_SUPP_NPP_INV', 'inv_softclip_mapping'),
        ('C_PP', 'inv_softclip_mapping'),
        ('D_SUPP_PP_INV', 'inv_softclip_mapping'),
        ('D_NPP', 'inv_softclip_mapping'),
        ('D_SUPP_NPP_LEFT', 'inv_softclip_mapping'),
        ('D_NPP_RIGHT_INV', 'C_SUPP_NPP_RIGHT'),
        ('C_SUPP_NPP_RIGHT', 'inv_softclip_mapping'),
        ('D_NPP_RIGHT_INV',),
        ('D_SUPP_NPP_RIGHT', 'D_NPP_RIGHT_INV'),
        ('D_NPP_RIGHT_INV', 'inv_softclip_mapping'),
        ('D_SUPP_NPP_INV', 'D_NPP_RIGHT_INV', 'C_SUPP_NPP_RIGHT'),
        ('D_SUPP_NPP_INV', 'D_NPP_RIGHT_INV'),
        ('C_SUPP_NPP_RIGHT',),
        ('D_SUPP_NPP_RIGHT', 'inv_softclip_mapping'),
        ('translocation_depth'),
        ('normal_depth_left'),
        ('normal_depth_right'),
        ('VAF_left'),
        ('VAF_right'),
        ('max_VAF')
    ]
    
    # STEP 6: Final sort for grouping
    df_sorted = df_with_groups.sort_values(
        by=['SORT_ORDER', 'GENE_PAIR', 'subgroup_size', 'subgroup', 'LEFT COORDINATE'],
        ascending=[True, True, False, True, True]
    ).reset_index(drop=True)
    
    # STEP 7: Build summary and output
    summary_blocks = []
    ungrouped_blocks = []
    
    # Define summary columns to sum
    summary_cols = [col for col in df.columns if col not in [
        'GROUP TAG', 'row_id', 'subgroup', 'subgroup_size',
        'LEFT COORDINATE', 'RIGHT COORDINATE', 'FIRST CHROMOSOME', 'SECOND CHROMOSOME',
        'LEFT GENE', 'RIGHT GENE', 'GENE_PAIR', 'TYPE', 'NOTATION'
    ]]
    
    for subgroup_id, group in df_sorted.groupby('subgroup', sort=False):
        # group = group.copy()
        group = group.sort_values(by='translocation_depth', ascending=False, na_position='last').copy()
        group_size = len(group)
        # summary_label = f"{group.iloc[0]['NOTATION']} {group.iloc[0]['LEFT GENE']}::{group.iloc[0]['RIGHT GENE']} {group.iloc[0]['TYPE']}"
        summary_label = f"GROUP: {group.iloc[0]['NOTATION']} {group.iloc[0]['LEFT GENE']}::{group.iloc[0]['RIGHT GENE']}"
    
        left_chr = extract_chromosome(group.iloc[0]["LEFT SIDE (5') OF NON-INVERTED BREAKPOINT"])
        right_chr = extract_chromosome(group.iloc[0]["RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT"])
        
        left_range = f"chr{left_chr} {int(group['LEFT COORDINATE'].min())}-{int(group['LEFT COORDINATE'].max())}"
        right_range = f"chr{right_chr} {int(group['RIGHT COORDINATE'].min())}-{int(group['RIGHT COORDINATE'].max())}"
    
        if group_size > 1:
            summary_row = {
                'GROUP TAG': summary_label,
                'NOTATION': '',
                'LEFT GENE': '',
                'RIGHT GENE': '',
                'TYPE': '',
                "LEFT SIDE (5') OF NON-INVERTED BREAKPOINT": left_range,
                "RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT": right_range,
                'row_id': '',
                'subgroup': subgroup_id,
                'subgroup_size': group_size,
            }
            
            for col in summary_cols:
                if col in group.columns:
                    if col not in ["LEFT SIDE (5') OF NON-INVERTED BREAKPOINT", "RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT"]:
                        summary_row[col] = pd.to_numeric(group[col], errors='coerce').sum()
    
            group['GROUP TAG'] = ''
            summary_blocks.append(pd.DataFrame([summary_row]))
            summary_blocks.append(group)
    
            blank_row = pd.DataFrame([{col: '' for col in group.columns}])
            summary_blocks.append(blank_row)
    
        else:
            group['GROUP TAG'] = ''
            ungrouped_blocks.append(group)
    
    # print(summary_blocks)
    
    # Add ungrouped rows if present
    if ungrouped_blocks:
        ungrouped = pd.concat(ungrouped_blocks, ignore_index=True)
        ungrouped = ungrouped.sort_values(by='translocation_depth', ascending=False, na_position='last')
    
        ungrouped_summary = pd.DataFrame([{
            'GROUP TAG': 'UNGROUPED',
            **{col: '' for col in ungrouped.columns if col != 'GROUP TAG'}
        }])
    
        summary_blocks.append(ungrouped_summary)
        summary_blocks.append(ungrouped)
    
    # STEP 8: Finalize output
    final_df = pd.concat(summary_blocks, ignore_index=True)
    
    # Make GROUP TAG the first column
    cols = ['GROUP TAG'] + [col for col in final_df.columns if col != 'GROUP TAG']
    final_df = final_df[cols]
    
    # Column order preference
    fixed_columns = ['GROUP TAG', 'NOTATION', "LEFT SIDE (5') OF NON-INVERTED BREAKPOINT", "RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT",
                     'TYPE', 'translocation_depth', 'normal_depth_left', 'normal_depth_right', 'max_normal_depth',
                     'VAF_left', 'VAF_right', 'max_VAF']
    other_columns = [col for col in final_df.columns if col not in fixed_columns]
    new_order = fixed_columns + other_columns
    final_df = final_df[new_order]
    final_df = final_df.drop(columns=['row_id', 'subgroup', 'SORT_ORDER', 'subgroup_size'], errors='ignore')
    
    final_df.to_csv(sample_name+'/'+sample_name+'_translocations_found_just_those_specifically_targeted_both_sides_panel_grouped_and_filtered.csv', index=False)
    
    # Save final_df to Excel with formatting
    output_file = sample_name+'/'+sample_name+'_translocations_found_just_those_specifically_targeted_both_sides_panel_grouped_and_filtered.xlsx'
    
    with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
        final_df.to_excel(writer, index=False, sheet_name='Summary')
    
        workbook  = writer.book
        worksheet = writer.sheets['Summary']
    
        # Freeze the top row
        worksheet.freeze_panes(1, 0)
    
        # Autosize all columns
        for i, col in enumerate(final_df.columns):
            # Find length of longest entry in the column (including header)
            series = final_df[col].astype(str)
            max_len = max(series.map(len).max(), len(str(col)))  # +2 for padding
            worksheet.set_column(i, i, min(max_len, 35))
    
        # Narrower widths for P to BC
        for col_idx in range(15, 55):
            worksheet.set_column(col_idx, col_idx, 8)
    
        # Define a bold and shaded format
        summary_format = workbook.add_format({
            'bold': True,
            'bg_color': '#D9E1F2',  # light blue shade
            'border': 0
        })
    
        # Loop through rows and apply format to summary rows
        for idx, value in enumerate(final_df['GROUP TAG']):
            if pd.notna(value) and value != '':
                worksheet.set_row(idx + 1, None, summary_format)  # +1 to account for header row
    
    return final_df

In [ ]:
def filter_and_group_chromosomal_rearrangements_2021_files(sample_name):
    df = pd.read_csv('UKCTOCS_testing/'+sample_name+'_translocations_found_just_those_specifically_targeted.csv')
    df['LEFT COORDINATE'] = df["LEFT SIDE (5') OF BREAKPOINT"].apply(extract_position2)
    df['RIGHT COORDINATE'] = df["RIGHT SIDE (3') OF BREAKPOINT"].apply(extract_position2)

    df = df.dropna(subset=['LEFT COORDINATE', 'RIGHT COORDINATE'])

    df['LEFT CHROMOSOME'] = df["LEFT SIDE (5') OF BREAKPOINT"].apply(extract_chromosome)
    df['RIGHT CHROMOSOME'] = df["RIGHT SIDE (3') OF BREAKPOINT"].apply(extract_chromosome)

    df['NOTATION'] = 't('+df['LEFT CHROMOSOME']+';'+df['RIGHT CHROMOSOME']
    df['TYPE'] = 'not available'
    
    df = df[df['LEFT GENE'] != df['RIGHT GENE']] #remove rows where the rearrangement supposedly occurs within the same gene
    df['max_normal_depth'] = maximum_of_two_columns(df['normal_depth_left'], df['normal_depth_right'])
    df['max_VAF'] = maximum_of_two_columns(df['VAF_left'], df['VAF_right'])
    
    # STEP 1: Reset and label rows
    df = df.reset_index(drop=True)  # start fresh
    df['row_id'] = df.index         # assign a unique integer row ID
    
    df['GENE_PAIR'] = fusion_genes(df['LEFT GENE'], df['RIGHT GENE'])
    
    # STEP 2: Sort
    df = df.sort_values(by=['NOTATION', 'GENE_PAIR', 'LEFT GENE', 'RIGHT GENE', 'TYPE', 'LEFT COORDINATE', 'RIGHT COORDINATE'])
    
    # STEP 3: Build groups
    grouped_blocks = []
    group_id_base = 0
    
    for _, group in df.groupby(['NOTATION', 'LEFT GENE', 'RIGHT GENE', 'TYPE']):
        g = group.copy()
        G = nx.Graph()    
        row_ids = g['row_id'].tolist()
        G.add_nodes_from(row_ids)
        g['subgroup'] = np.nan
    
        for i, j in combinations(g.index, 2):
            row_i = g.loc[i]
            row_j = g.loc[j]
            if (
                abs(row_i['LEFT COORDINATE'] - row_j['LEFT COORDINATE']) <= 500
                and abs(row_i['RIGHT COORDINATE'] - row_j['RIGHT COORDINATE']) <= 500
            ):
                G.add_edge(row_i['row_id'], row_j['row_id'])
    
        for group_num, component in enumerate(nx.connected_components(G), start=group_id_base):
            idx = g['row_id'].isin(component)
            g.loc[idx, 'subgroup'] = group_num
    
        group_id_base = int(g['subgroup'].max()) + 1
        grouped_blocks.append(g)
    
    # STEP 4: Combine and annotate group sizes
    df_with_groups = pd.concat(grouped_blocks).reset_index(drop=True)
    df_with_groups['subgroup'] = df_with_groups['subgroup'].astype(int)
    group_sizes = df_with_groups['subgroup'].value_counts()
    df_with_groups['subgroup_size'] = df_with_groups['subgroup'].map(group_sizes)
    
    # STEP 5: Rank gene pairs by frequency
    gene_pair_sizes = df_with_groups['GENE_PAIR'].value_counts()
    gene_pair_rank = {gp: i for i, gp in enumerate(gene_pair_sizes.index, start=1)}
    df_with_groups['SORT_ORDER'] = df_with_groups['GENE_PAIR'].map(gene_pair_rank)
    
    summary_cols = [
        'D_NO_OVERLAP',
        ('D_NPP', 'C_SUPP_NPP'),
        ('D_NPP',),
        ('D_NPP', 'softclip_mapping'),
        ('D_SUPP_PP', 'C_PP'),
        ('D_SUPP_NPP', 'D_NPP'),
        ('concordant_1_end_mapping',),
        ('D_NPP_MPP', 'softclip_mapping'),
        ('C_SUPP_NPP', 'softclip_mapping'),
        ('D_SUPP_NPP', 'D_NPP_MPP'),
        ('C_PP',),
        ('D_SUPP_NPP', 'D_NPP', 'C_SUPP_NPP'),
        ('D_SUPP_NPP', 'C_NPP'),
        ('D_SUPP_NPP', 'C_SUPP_NPP'),
        ('D_SUPP_NPP', 'softclip_mapping'),
        ('D_NPP_MPP', 'C_SUPP_NPP'),
        ('C_PP', 'softclip_mapping'),
        ('C_NPP', 'softclip_mapping'),
        ('D_NPP_LEFT_INV', 'C_SUPP_NPP_LEFT'),
        ('D_NPP_LEFT_INV',),
        ('D_NPP_LEFT_INV', 'inv_softclip_mapping'),
        ('D_SUPP_NPP_LEFT', 'D_NPP_LEFT_INV'),
        ('D_SUPP_PP_INV', 'C_PP'),
        'D_NO_OVERLAP_INV',
        ('D_SUPP_NPP_INV', 'D_NPP'),
        ('D_SUPP_NPP_INV', 'D_NPP_LEFT_INV'),
        ('D_SUPP_NPP_INV', 'inv_softclip_mapping'),
        ('C_PP', 'inv_softclip_mapping'),
        ('D_SUPP_PP_INV', 'inv_softclip_mapping'),
        ('D_NPP', 'inv_softclip_mapping'),
        ('D_SUPP_NPP_LEFT', 'inv_softclip_mapping'),
        ('D_NPP_RIGHT_INV', 'C_SUPP_NPP_RIGHT'),
        ('C_SUPP_NPP_RIGHT', 'inv_softclip_mapping'),
        ('D_NPP_RIGHT_INV',),
        ('D_SUPP_NPP_RIGHT', 'D_NPP_RIGHT_INV'),
        ('D_NPP_RIGHT_INV', 'inv_softclip_mapping'),
        ('D_SUPP_NPP_INV', 'D_NPP_RIGHT_INV', 'C_SUPP_NPP_RIGHT'),
        ('D_SUPP_NPP_INV', 'D_NPP_RIGHT_INV'),
        ('C_SUPP_NPP_RIGHT',),
        ('D_SUPP_NPP_RIGHT', 'inv_softclip_mapping'),
        ('translocation_depth'),
        ('normal_depth_left'),
        ('normal_depth_right'),
        ('VAF_left'),
        ('VAF_right'),
        ('max_VAF')
    ]
    
    # STEP 6: Final sort for grouping
    df_sorted = df_with_groups.sort_values(
        by=['SORT_ORDER', 'GENE_PAIR', 'subgroup_size', 'subgroup', 'LEFT COORDINATE'],
        ascending=[True, True, False, True, True]
    ).reset_index(drop=True)
    
    # STEP 7: Build summary and output
    summary_blocks = []
    ungrouped_blocks = []
    
    # Define summary columns to sum
    summary_cols = [col for col in df.columns if col not in [
        'GROUP TAG', 'row_id', 'subgroup', 'subgroup_size',
        'LEFT COORDINATE', 'RIGHT COORDINATE', 'FIRST CHROMOSOME', 'SECOND CHROMOSOME',
        'LEFT GENE', 'RIGHT GENE', 'GENE_PAIR', 'TYPE', 'NOTATION'
    ]]
    
    for subgroup_id, group in df_sorted.groupby('subgroup', sort=False):
        # group = group.copy()
        group = group.sort_values(by='translocation_depth', ascending=False, na_position='last').copy()
        group_size = len(group)
        # summary_label = f"{group.iloc[0]['NOTATION']} {group.iloc[0]['LEFT GENE']}::{group.iloc[0]['RIGHT GENE']} {group.iloc[0]['TYPE']}"
        summary_label = f"GROUP: {group.iloc[0]['NOTATION']} {group.iloc[0]['LEFT GENE']}::{group.iloc[0]['RIGHT GENE']}"
    
        left_chr = extract_chromosome(group.iloc[0]["LEFT SIDE (5') OF BREAKPOINT"])
        right_chr = extract_chromosome(group.iloc[0]["RIGHT SIDE (3') OF BREAKPOINT"])
        
        left_range = f"chr{left_chr} {int(group['LEFT COORDINATE'].min())}-{int(group['LEFT COORDINATE'].max())}"
        right_range = f"chr{right_chr} {int(group['RIGHT COORDINATE'].min())}-{int(group['RIGHT COORDINATE'].max())}"
    
        if group_size > 1:
            summary_row = {
                'GROUP TAG': summary_label,
                'NOTATION': '',
                'LEFT GENE': '',
                'RIGHT GENE': '',
                'TYPE': '',
                "LEFT SIDE (5') OF BREAKPOINT": left_range,
                "RIGHT SIDE (3') OF BREAKPOINT": right_range,
                'row_id': '',
                'subgroup': subgroup_id,
                'subgroup_size': group_size,
            }
            
            for col in summary_cols:
                if col in group.columns:
                    if col not in ["LEFT SIDE (5') OF BREAKPOINT", "RIGHT SIDE (3') OF BREAKPOINT"]:
                        summary_row[col] = pd.to_numeric(group[col], errors='coerce').sum()
    
            group['GROUP TAG'] = ''
            summary_blocks.append(pd.DataFrame([summary_row]))
            summary_blocks.append(group)
    
            blank_row = pd.DataFrame([{col: '' for col in group.columns}])
            summary_blocks.append(blank_row)
    
        else:
            group['GROUP TAG'] = ''
            ungrouped_blocks.append(group)
    
    # print(summary_blocks)
    
    # Add ungrouped rows if present
    if ungrouped_blocks:
        ungrouped = pd.concat(ungrouped_blocks, ignore_index=True)
        ungrouped = ungrouped.sort_values(by='translocation_depth', ascending=False, na_position='last')
    
        ungrouped_summary = pd.DataFrame([{
            'GROUP TAG': 'UNGROUPED',
            **{col: '' for col in ungrouped.columns if col != 'GROUP TAG'}
        }])
    
        summary_blocks.append(ungrouped_summary)
        summary_blocks.append(ungrouped)
    
    # STEP 8: Finalize output
    final_df = pd.concat(summary_blocks, ignore_index=True)
    
    # Make GROUP TAG the first column
    cols = ['GROUP TAG'] + [col for col in final_df.columns if col != 'GROUP TAG']
    final_df = final_df[cols]
    
    # Column order preference
    fixed_columns = ['GROUP TAG', 'NOTATION', "LEFT SIDE (5') OF BREAKPOINT", "RIGHT SIDE (3') OF BREAKPOINT",
                     'TYPE', 'translocation_depth', 'normal_depth_left', 'normal_depth_right', 'max_normal_depth',
                     'VAF_left', 'VAF_right', 'max_VAF']
    other_columns = [col for col in final_df.columns if col not in fixed_columns]
    new_order = fixed_columns + other_columns
    final_df = final_df[new_order]
    final_df = final_df.drop(columns=['row_id', 'subgroup', 'SORT_ORDER', 'subgroup_size'], errors='ignore')
    
    final_df.to_csv('UKCTOCS_testing/'+sample_name+'_translocations_found_just_those_specifically_targeted_grouped_and_filtered.csv', index=False)
    
    # Save final_df to Excel with formatting
    output_file = 'UKCTOCS_testing/'+sample_name+'_translocations_found_just_those_specifically_targeted_grouped_and_filtered.xlsx'
    
    with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
        final_df.to_excel(writer, index=False, sheet_name='Summary')
    
        workbook  = writer.book
        worksheet = writer.sheets['Summary']
    
        # Freeze the top row
        worksheet.freeze_panes(1, 0)
    
        # Autosize all columns
        for i, col in enumerate(final_df.columns):
            # Find length of longest entry in the column (including header)
            series = final_df[col].astype(str)
            max_len = max(series.map(len).max(), len(str(col)))  # +2 for padding
            worksheet.set_column(i, i, min(max_len, 35))
    
        # Narrower widths for P to BC
        for col_idx in range(15, 55):
            worksheet.set_column(col_idx, col_idx, 8)
    
        # Define a bold and shaded format
        summary_format = workbook.add_format({
            'bold': True,
            'bg_color': '#D9E1F2',  # light blue shade
            'border': 0
        })
    
        # Loop through rows and apply format to summary rows
        for idx, value in enumerate(final_df['GROUP TAG']):
            if pd.notna(value) and value != '':
                worksheet.set_row(idx + 1, None, summary_format)  # +1 to account for header row
    
    return final_df

In [ ]:
def filter_and_group_chromosomal_rearrangements_v2(folder, sample_name):
    df = pd.read_csv(folder+'/'+sample_name+'_translocations_found_just_those_specifically_targeted_both_sides_panel.csv')
    df['LEFT COORDINATE'] = df["LEFT SIDE (5') OF NON-INVERTED BREAKPOINT"].apply(extract_position)
    df['RIGHT COORDINATE'] = df["RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT"].apply(extract_position)
    df = df[df['LEFT GENE'] != df['RIGHT GENE']] #remove rows where the rearrangement supposedly occurs within the same gene
    df['max_normal_depth'] = maximum_of_two_columns(df['normal_depth_left'], df['normal_depth_right'])
    df['max_VAF'] = maximum_of_two_columns(df['VAF_left'], df['VAF_right'])
    
    # STEP 1: Reset and label rows
    df = df.reset_index(drop=True)  # start fresh
    df['row_id'] = df.index         # assign a unique integer row ID
    
    df['GENE_PAIR'] = fusion_genes(df['LEFT GENE'], df['RIGHT GENE'])
    
    # STEP 2: Sort
    df = df.sort_values(by=['NOTATION', 'GENE_PAIR', 'LEFT GENE', 'RIGHT GENE', 'TYPE', 'LEFT COORDINATE', 'RIGHT COORDINATE'])
    
    # STEP 3: Build groups
    grouped_blocks = []
    group_id_base = 0
    
    for _, group in df.groupby(['NOTATION', 'LEFT GENE', 'RIGHT GENE', 'TYPE']):
        g = group.copy()
        G = nx.Graph()    
        row_ids = g['row_id'].tolist()
        G.add_nodes_from(row_ids)
        g['subgroup'] = np.nan
    
        for i, j in combinations(g.index, 2):
            row_i = g.loc[i]
            row_j = g.loc[j]
            if (
                abs(row_i['LEFT COORDINATE'] - row_j['LEFT COORDINATE']) <= 500
                and abs(row_i['RIGHT COORDINATE'] - row_j['RIGHT COORDINATE']) <= 500
            ):
                G.add_edge(row_i['row_id'], row_j['row_id'])
    
        for group_num, component in enumerate(nx.connected_components(G), start=group_id_base):
            idx = g['row_id'].isin(component)
            g.loc[idx, 'subgroup'] = group_num
    
        group_id_base = int(g['subgroup'].max()) + 1
        grouped_blocks.append(g)
    
    # STEP 4: Combine and annotate group sizes
    df_with_groups = pd.concat(grouped_blocks).reset_index(drop=True)
    df_with_groups['subgroup'] = df_with_groups['subgroup'].astype(int)
    group_sizes = df_with_groups['subgroup'].value_counts()
    df_with_groups['subgroup_size'] = df_with_groups['subgroup'].map(group_sizes)
    
    # STEP 5: Rank gene pairs by frequency
    gene_pair_sizes = df_with_groups['GENE_PAIR'].value_counts()
    gene_pair_rank = {gp: i for i, gp in enumerate(gene_pair_sizes.index, start=1)}
    df_with_groups['SORT_ORDER'] = df_with_groups['GENE_PAIR'].map(gene_pair_rank)
    
    summary_cols = [
        'D_NO_OVERLAP',
        ('D_NPP', 'C_SUPP_NPP'),
        ('D_NPP',),
        ('D_NPP', 'softclip_mapping'),
        ('D_SUPP_PP', 'C_PP'),
        ('D_SUPP_NPP', 'D_NPP'),
        ('concordant_1_end_mapping',),
        ('D_NPP_MPP', 'softclip_mapping'),
        ('C_SUPP_NPP', 'softclip_mapping'),
        ('D_SUPP_NPP', 'D_NPP_MPP'),
        ('C_PP',),
        ('D_SUPP_NPP', 'D_NPP', 'C_SUPP_NPP'),
        ('D_SUPP_NPP', 'C_NPP'),
        ('D_SUPP_NPP', 'C_SUPP_NPP'),
        ('D_SUPP_NPP', 'softclip_mapping'),
        ('D_NPP_MPP', 'C_SUPP_NPP'),
        ('C_PP', 'softclip_mapping'),
        ('C_NPP', 'softclip_mapping'),
        ('D_NPP_LEFT_INV', 'C_SUPP_NPP_LEFT'),
        ('D_NPP_LEFT_INV',),
        ('D_NPP_LEFT_INV', 'inv_softclip_mapping'),
        ('D_SUPP_NPP_LEFT', 'D_NPP_LEFT_INV'),
        ('D_SUPP_PP_INV', 'C_PP'),
        'D_NO_OVERLAP_INV',
        ('D_SUPP_NPP_INV', 'D_NPP'),
        ('D_SUPP_NPP_INV', 'D_NPP_LEFT_INV'),
        ('D_SUPP_NPP_INV', 'inv_softclip_mapping'),
        ('C_PP', 'inv_softclip_mapping'),
        ('D_SUPP_PP_INV', 'inv_softclip_mapping'),
        ('D_NPP', 'inv_softclip_mapping'),
        ('D_SUPP_NPP_LEFT', 'inv_softclip_mapping'),
        ('D_NPP_RIGHT_INV', 'C_SUPP_NPP_RIGHT'),
        ('C_SUPP_NPP_RIGHT', 'inv_softclip_mapping'),
        ('D_NPP_RIGHT_INV',),
        ('D_SUPP_NPP_RIGHT', 'D_NPP_RIGHT_INV'),
        ('D_NPP_RIGHT_INV', 'inv_softclip_mapping'),
        ('D_SUPP_NPP_INV', 'D_NPP_RIGHT_INV', 'C_SUPP_NPP_RIGHT'),
        ('D_SUPP_NPP_INV', 'D_NPP_RIGHT_INV'),
        ('C_SUPP_NPP_RIGHT',),
        ('D_SUPP_NPP_RIGHT', 'inv_softclip_mapping'),
        ('translocation_depth'),
        ('normal_depth_left'),
        ('normal_depth_right'),
        ('VAF_left'),
        ('VAF_right'),
        ('max_VAF')
    ]
    
    # STEP 6: Final sort for grouping
    df_sorted = df_with_groups.sort_values(
        by=['SORT_ORDER', 'GENE_PAIR', 'subgroup_size', 'subgroup', 'LEFT COORDINATE'],
        ascending=[True, True, False, True, True]
    ).reset_index(drop=True)

    # STEP 7: Build summary and output
    summary_blocks = []
    ungrouped_blocks = []
    
    # Define summary columns to sum
    summary_cols = [col for col in df.columns if col not in [
        'GROUP TAG', 'row_id', 'subgroup', 'subgroup_size',
        'LEFT COORDINATE', 'RIGHT COORDINATE', 'FIRST CHROMOSOME', 'SECOND CHROMOSOME',
        'LEFT GENE', 'RIGHT GENE', 'GENE_PAIR', 'TYPE', 'NOTATION'
    ]]
    
    for subgroup_id, group in df_sorted.groupby('subgroup', sort=False):
        # group = group.copy()
        group = group.sort_values(by='translocation_depth', ascending=False, na_position='last').copy()
        group_size = len(group)
        # summary_label = f"{group.iloc[0]['NOTATION']} {group.iloc[0]['LEFT GENE']}::{group.iloc[0]['RIGHT GENE']} {group.iloc[0]['TYPE']}"
        summary_label = f"GROUP: {group.iloc[0]['NOTATION']} {group.iloc[0]['LEFT GENE']}::{group.iloc[0]['RIGHT GENE']}"
        notation = group.iloc[0]['NOTATION']
        left_gene = group.iloc[0]['LEFT GENE']
        right_gene = group.iloc[0]['RIGHT GENE']
        rearr_type = group.iloc[0]['TYPE']
        gene_pair = group.iloc[0]['GENE_PAIR']
    
        left_chr = extract_chromosome(group.iloc[0]["LEFT SIDE (5') OF NON-INVERTED BREAKPOINT"])
        right_chr = extract_chromosome(group.iloc[0]["RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT"])
        
        left_range = f"chr{left_chr} {int(group['LEFT COORDINATE'].min())}-{int(group['LEFT COORDINATE'].max())}"
        right_range = f"chr{right_chr} {int(group['RIGHT COORDINATE'].min())}-{int(group['RIGHT COORDINATE'].max())}"
    
        if group_size > 1:
            summary_row = {
                'GROUP TAG': summary_label,
                'NOTATION': notation,
                'LEFT GENE': left_gene,
                'RIGHT GENE': right_gene,
                'TYPE': rearr_type,
                "LEFT SIDE (5') OF NON-INVERTED BREAKPOINT": left_range,
                "RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT": right_range,
                'row_id': '',
                'subgroup': subgroup_id,
                'subgroup_size': group_size,
            }
            
            for col in summary_cols:
                if col in group.columns:
                    if col not in ["NOTATION", "TYPE", "LEFT GENE", "RIGHT GENE", "LEFT SIDE (5') OF NON-INVERTED BREAKPOINT", "RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT"]:
                        summary_row[col] = pd.to_numeric(group[col], errors='coerce').sum()
    
            group['GROUP TAG'] = ''
            summary_blocks.append(pd.DataFrame([summary_row]))
            summary_blocks.append(group)
    
            blank_row = pd.DataFrame([{col: '' for col in group.columns}])
            summary_blocks.append(blank_row)
    
        else:
            group['GROUP TAG'] = ''
            ungrouped_blocks.append(group)

    # Add ungrouped rows if present
    if ungrouped_blocks:
        ungrouped = pd.concat(ungrouped_blocks, ignore_index=True)
        ungrouped = ungrouped.sort_values(by='translocation_depth', ascending=False, na_position='last')
    
        # Define GENE_PAIRs of interest
        pairs_of_interest = {
            'MYH11::CBFB', 'CBFB::MYH11',
            'RUNX1::RUNX1T1', 'RUNX1T1::RUNX1',
            'DEK::NUP214', 'NUP214::DEK',
            'KMT2A::MLLT3', 'MLLT3::KMT2A',
            'BCR::ABL1', 'ABL1::BCR',
            'PML::RARA', 'RARA::PML'
        }
    
        # Split ungrouped into two sets
        ungrouped_of_interest = ungrouped[ungrouped['GENE_PAIR'].isin(pairs_of_interest)].copy()
        ungrouped_rest = ungrouped[~ungrouped['GENE_PAIR'].isin(pairs_of_interest)].copy()
    
        # Sort the 'of interest' set by GENE_PAIR then translocation_depth
        ungrouped_of_interest = ungrouped_of_interest.sort_values(
            by=['GENE_PAIR', 'translocation_depth'], ascending=[True, False]
        )
    
        # Add label and rows for "ungrouped, but potentially of interest"
        if not ungrouped_of_interest.empty:
            blank_row = pd.DataFrame([{col: '' for col in group.columns}])
            ungrouped_interest_summary = pd.DataFrame([{
                'GROUP TAG': 'UNGROUPED, BUT POTENTIALLY OF INTEREST',
                **{col: '' for col in ungrouped_of_interest.columns if col != 'GROUP TAG'}
            }])
            summary_blocks.append(ungrouped_interest_summary)
            summary_blocks.append(ungrouped_of_interest)
            summary_blocks.append(blank_row)
    
        # Add label and rows for the remaining ungrouped
        if not ungrouped_rest.empty:
            ungrouped_summary = pd.DataFrame([{
                'GROUP TAG': 'UNGROUPED',
                **{col: '' for col in ungrouped_rest.columns if col != 'GROUP TAG'}
            }])
            summary_blocks.append(ungrouped_summary)
            summary_blocks.append(ungrouped_rest)
    
    # STEP 8: Finalize output
    final_df = pd.concat(summary_blocks, ignore_index=True)

    # # Count number of non-zero and non-empty entries across columns O to BC
    # evidence_cols = final_df.columns[14:55]
    # final_df['evidence_count'] = final_df[evidence_cols].apply(
    #     lambda row: sum(pd.to_numeric(row, errors='coerce').fillna(0) > 0),
    #     axis=1
    # )

    # Determine the dynamic range of evidence columns (those between 'max_VAF' and 'LEFT GENE' in the original input)
    input_df = pd.read_csv(folder+'/'+sample_name+'_translocations_found_just_those_specifically_targeted_both_sides_panel.csv')
    col_list = list(input_df.columns)
    max_vaf_idx = col_list.index('max_VAF')  # should exist after your processing    
    left_gene_idx = col_list.index('LEFT GENE')
    evidence_cols = col_list[max_vaf_idx+1:left_gene_idx]
    
    # Now apply the count across these evidence columns
    evidence_cols = [col for col in evidence_cols if col in final_df.columns]  # ensure all exist
    final_df['evidence_count'] = final_df[evidence_cols].apply(
        lambda row: sum(pd.to_numeric(row, errors='coerce').fillna(0) > 0),
        axis=1
    )
    
    # Make GROUP TAG the first column
    cols = ['GROUP TAG'] + [col for col in final_df.columns if col != 'GROUP TAG']
    final_df = final_df[cols]
    
    # Column order preference
    fixed_columns = ['GROUP TAG', 'NOTATION', "LEFT SIDE (5') OF NON-INVERTED BREAKPOINT", "RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT",
                     'TYPE', 'translocation_depth', 'normal_depth_left', 'normal_depth_right', 'max_normal_depth',
                     'VAF_left', 'VAF_right', 'max_VAF', 'evidence_count']
    other_columns = [col for col in final_df.columns if col not in fixed_columns]
    new_order = fixed_columns + other_columns
    final_df = final_df[new_order]
    final_df = final_df.drop(columns=['row_id', 'subgroup', 'SORT_ORDER', 'subgroup_size'], errors='ignore')
    
    final_df.to_csv(folder+'/'+sample_name+'_translocations_found_just_those_specifically_targeted_both_sides_panel_grouped_and_filtered_v2.csv', index=False)
    
    # Save final_df to Excel with formatting
    output_file = folder+'/'+sample_name+'_translocations_found_just_those_specifically_targeted_both_sides_panel_grouped_and_filtered_v2.xlsx'
    
    with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
        final_df.to_excel(writer, index=False, sheet_name='Summary')
    
        workbook  = writer.book
        worksheet = writer.sheets['Summary']
    
        # Freeze the top row
        worksheet.freeze_panes(1, 0)
    
        # Autosize all columns
        for i, col in enumerate(final_df.columns):
            # Find length of longest entry in the column (including header)
            series = final_df[col].astype(str)
            max_len = max(series.map(len).max(), len(str(col)))  # +2 for padding
            worksheet.set_column(i, i, min(max_len, 35))
    
        # Narrower widths for P to BC
        for col_idx in range(15, 55):
            worksheet.set_column(col_idx, col_idx, 8)
    
        # # Define a bold and shaded format
        # summary_format = workbook.add_format({
        #     'bold': True,
        #     'bg_color': '#D9E1F2',  # light blue shade
        #     'border': 0
        # })
    
        # # Loop through rows and apply format to summary rows
        # for idx, value in enumerate(final_df['GROUP TAG']):
        #     if pd.notna(value) and value != '':
        #         worksheet.set_row(idx + 1, None, summary_format)  # +1 to account for header row


        # Define formatting styles
        group_summary_format = workbook.add_format({
            'bold': True,
            'bg_color': '#D9E1F2',  # light blue
            'border': 0
        })
        
        ungrouped_summary_format = workbook.add_format({
            'bold': True,
            'bg_color': '#FCE4D6',  # light orange-peach
            'border': 0
        })
        
        # Loop through rows and apply conditional formatting
        for idx, value in enumerate(final_df['GROUP TAG']):
            if pd.notna(value) and value != '':
                if 'UNGROUPED' in value.upper():
                    worksheet.set_row(idx + 1, None, ungrouped_summary_format)  # offset by 1 for header
                else:
                    worksheet.set_row(idx + 1, None, group_summary_format)

    
    return final_df

In [ ]:
# Step 1: Define expected inversion status for each gene pair
expected_inversion = {
    'KMT2A::MLLT3': True,
    'MLLT3::KMT2A': True,
    'BCR::ABL1': False,
    'ABL1::BCR': False,
    'PML::RARA': False,
    'RARA::PML': False,
    'RUNX1::RUNX1T1': False,
    'RUNX1T1::RUNX1': False,
    'DEK::NUP214': True,
    'NUP214::DEK': True,
    # 'CBFB::MYH11': True,   # inv(16)
    # 'MYH11::CBFB': True,   # inv(16)
    # Include t(16;16) versions if needed:
    # 'CBFB::MYH11_t1616': False,
    # 'MYH11::CBFB_t1616': False,
}

# Function to match based on inversion presence in GROUPED TYPE
def matches_expected_inversion(row):
    gene_pair = row['GENE_PAIR']
    grouped_type = row['GROUPED TYPE']
    
    expected = expected_inversion.get(gene_pair)

    if expected is None:
        # Option: return True if you want to keep unknowns, or False to exclude them
        return True

    # Extract inversion presence from text
    has_inversion = 'WITH INVERTED SEGMENT' in grouped_type
    has_no_inversion = 'WITH NO INVERTED SEGMENT' in grouped_type

    # Compare observed to expected
    if expected and has_inversion:
        # print(gene_pair)
        # print(grouped_type)
        # print('expected = ', expected)
        # print('has inversion = ', has_inversion)
        return True
    if not expected and has_no_inversion:
        # print(gene_pair)
        # print(grouped_type)
        # print('not expected = ')
        # print('has no inversion = ', has_no_inversion)
        return True
    
    return False  # mismatch

In [ ]:
def filter_and_group_chromosomal_rearrangements_v3(folder, sample_name):

    df = pd.read_csv(folder+'/'+sample_name+'_translocations_found_just_those_specifically_targeted_both_sides_panel.csv')
    df['LEFT COORDINATE'] = df["LEFT SIDE (5') OF NON-INVERTED BREAKPOINT"].apply(extract_position)
    df['RIGHT COORDINATE'] = df["RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT"].apply(extract_position)
    df = df[df['LEFT GENE'] != df['RIGHT GENE']] #remove rows where the rearrangement supposedly occurs within the same gene
    df['max_normal_depth'] = maximum_of_two_columns(df['normal_depth_left'], df['normal_depth_right'])
    df['max_VAF'] = maximum_of_two_columns(df['VAF_left'], df['VAF_right'])
    
    # Strip " on left" or " on right" from inversion descriptions
    df['GROUPED TYPE'] = df['TYPE'].astype(str).str.replace(
        r' ON (LEFT|RIGHT)', '', regex=True
    )
        
    # Determine the dynamic range of evidence columns (those between 'max_VAF' and 'LEFT GENE' in the original input)
    col_list = list(df.columns)
    max_vaf_idx = col_list.index('max_VAF')  # should exist after your processing    
    left_gene_idx = col_list.index('LEFT GENE')
    evidence_cols = col_list[max_vaf_idx+1:left_gene_idx]
    
    # Now apply the count across these evidence columns
    evidence_cols = [col for col in evidence_cols if col in df.columns]  # ensure all exist
    df['evidence_count'] = df[evidence_cols].apply(
        lambda row: sum(pd.to_numeric(row, errors='coerce').fillna(0) > 0),
        axis=1
    )

    # Define discordant read columns (start with any of: "D_", "'D_", "('D_")
    discordant_cols = [
        col for col in df.columns
        if isinstance(col, str) and (
            col.startswith("D_") or
            col.startswith("'D_") or
            col.startswith("('D_")
        )
    ]
    
    # Define split (concordant) read columns (start with: "C_", "'C_", "('C_", or "('concordant")
    split_cols = [
        col for col in df.columns
        if isinstance(col, str) and (
            col.startswith("C_") or
            col.startswith("'C_") or
            col.startswith("('C_") or
            col.startswith("('concordant")
        )
    ]
    
    # Create the new summed columns
    df["discordant reads"] = df[discordant_cols].sum(axis=1, skipna=True)
    df["split reads"] = df[split_cols].sum(axis=1, skipna=True)
    
    
    # STEP 1: Reset and label rows
    df = df.reset_index(drop=True)  # start fresh
    df['row_id'] = df.index         # assign a unique integer row ID
    df['GENE_PAIR'] = fusion_genes(df['LEFT GENE'], df['RIGHT GENE'])
    df['GROUP TAG'] = ''
        # Add a sample column if needed:
    df["SAMPLE NAME"] = sample_name

    # STEP 2: Sort
    df = df.sort_values(by=['NOTATION', 'GENE_PAIR', 'LEFT GENE', 'RIGHT GENE', 'GROUPED TYPE', 'LEFT COORDINATE', 'RIGHT COORDINATE'])
    
    # STEP 3: Build groups
    grouped_blocks = []
    group_id_base = 0
    
    for _, group in df.groupby(['NOTATION', 'LEFT GENE', 'RIGHT GENE', 'GROUPED TYPE']):
        g = group.copy()
        G = nx.Graph()    
        row_ids = g['row_id'].tolist()
        G.add_nodes_from(row_ids)
        g['subgroup'] = np.nan
    
        for i, j in combinations(g.index, 2):
            row_i = g.loc[i]
            row_j = g.loc[j]
            if (
                abs(row_i['LEFT COORDINATE'] - row_j['LEFT COORDINATE']) <= 500
                and abs(row_i['RIGHT COORDINATE'] - row_j['RIGHT COORDINATE']) <= 500
            ):
                G.add_edge(row_i['row_id'], row_j['row_id'])
    
        for group_num, component in enumerate(nx.connected_components(G), start=group_id_base):
            idx = g['row_id'].isin(component)
            g.loc[idx, 'subgroup'] = group_num
    
        group_id_base = int(g['subgroup'].max()) + 1
        grouped_blocks.append(g)
    
    # STEP 4: Combine and annotate group sizes
    df_with_groups = pd.concat(grouped_blocks).reset_index(drop=True)
    df_with_groups['subgroup'] = df_with_groups['subgroup'].astype(int)
    group_sizes = df_with_groups['subgroup'].value_counts()
    df_with_groups['subgroup_size'] = df_with_groups['subgroup'].map(group_sizes)
    
    # STEP 5: Rank gene pairs by frequency
    gene_pair_sizes = df_with_groups['GENE_PAIR'].value_counts()
    gene_pair_rank = {gp: i for i, gp in enumerate(gene_pair_sizes.index, start=1)}
    df_with_groups['SORT_ORDER'] = df_with_groups['GENE_PAIR'].map(gene_pair_rank)
    
    summary_cols = [
        'D_NO_OVERLAP',
        ('D_NPP', 'C_SUPP_NPP'),
        ('D_NPP',),
        ('D_NPP', 'softclip_mapping'),
        ('D_SUPP_PP', 'C_PP'),
        ('D_SUPP_NPP', 'D_NPP'),
        ('concordant_1_end_mapping',),
        ('D_NPP_MPP', 'softclip_mapping'),
        ('C_SUPP_NPP', 'softclip_mapping'),
        ('D_SUPP_NPP', 'D_NPP_MPP'),
        ('C_PP',),
        ('D_SUPP_NPP', 'D_NPP', 'C_SUPP_NPP'),
        ('D_SUPP_NPP', 'C_NPP'),
        ('D_SUPP_NPP', 'C_SUPP_NPP'),
        ('D_SUPP_NPP', 'softclip_mapping'),
        ('D_NPP_MPP', 'C_SUPP_NPP'),
        ('C_PP', 'softclip_mapping'),
        ('C_NPP', 'softclip_mapping'),
        ('D_NPP_LEFT_INV', 'C_SUPP_NPP_LEFT'),
        ('D_NPP_LEFT_INV',),
        ('D_NPP_LEFT_INV', 'inv_softclip_mapping'),
        ('D_SUPP_NPP_LEFT', 'D_NPP_LEFT_INV'),
        ('D_SUPP_PP_INV', 'C_PP'),
        'D_NO_OVERLAP_INV',
        ('D_SUPP_NPP_INV', 'D_NPP'),
        ('D_SUPP_NPP_INV', 'D_NPP_LEFT_INV'),
        ('D_SUPP_NPP_INV', 'inv_softclip_mapping'),
        ('C_PP', 'inv_softclip_mapping'),
        ('D_SUPP_PP_INV', 'inv_softclip_mapping'),
        ('D_NPP', 'inv_softclip_mapping'),
        ('D_SUPP_NPP_LEFT', 'inv_softclip_mapping'),
        ('D_NPP_RIGHT_INV', 'C_SUPP_NPP_RIGHT'),
        ('C_SUPP_NPP_RIGHT', 'inv_softclip_mapping'),
        ('D_NPP_RIGHT_INV',),
        ('D_SUPP_NPP_RIGHT', 'D_NPP_RIGHT_INV'),
        ('D_NPP_RIGHT_INV', 'inv_softclip_mapping'),
        ('D_SUPP_NPP_INV', 'D_NPP_RIGHT_INV', 'C_SUPP_NPP_RIGHT'),
        ('D_SUPP_NPP_INV', 'D_NPP_RIGHT_INV'),
        ('C_SUPP_NPP_RIGHT',),
        ('D_SUPP_NPP_RIGHT', 'inv_softclip_mapping'),
        ('translocation_depth'),
        ('normal_depth_left'),
        ('normal_depth_right'),
        ('VAF_left'),
        ('VAF_right'),
        ('max_VAF'),
        ('discordant reads'),
        ('split reads')
    ]
    
    # STEP 6: Final sort for grouping
    df_sorted = df_with_groups.sort_values(
        by=['SORT_ORDER', 'GENE_PAIR', 'subgroup_size', 'subgroup', 'LEFT COORDINATE'],
        ascending=[True, True, False, True, True]
    ).reset_index(drop=True)

    # STEP 7: Build summary and output
    summary_blocks = []
    ungrouped_blocks = []
    
    # Define summary columns to sum
    summary_cols = [col for col in df.columns if col not in [
        'GROUP TAG', 'row_id', 'subgroup', 'subgroup_size',
        'LEFT COORDINATE', 'RIGHT COORDINATE', 'FIRST CHROMOSOME', 'SECOND CHROMOSOME',
        'LEFT GENE', 'RIGHT GENE', 'GENE_PAIR', 'GROUPED TYPE', 'NOTATION'
    ]]
    
    for subgroup_id, group in df_sorted.groupby('subgroup', sort=False):
        # group = group.copy()
        group = group.sort_values(by='translocation_depth', ascending=False, na_position='last').copy()
        group_size = len(group)
        # summary_label = f"{group.iloc[0]['NOTATION']} {group.iloc[0]['LEFT GENE']}::{group.iloc[0]['RIGHT GENE']} {group.iloc[0]['TYPE']}"
        summary_label = f"GROUP: {group.iloc[0]['NOTATION']} {group.iloc[0]['LEFT GENE']}::{group.iloc[0]['RIGHT GENE']}"
        notation = group.iloc[0]['NOTATION']
        left_gene = group.iloc[0]['LEFT GENE']
        right_gene = group.iloc[0]['RIGHT GENE']
        rearr_type = group.iloc[0]['GROUPED TYPE']
        gene_pair = group.iloc[0]['GENE_PAIR']
    
        left_chr = extract_chromosome(group.iloc[0]["LEFT SIDE (5') OF NON-INVERTED BREAKPOINT"])
        right_chr = extract_chromosome(group.iloc[0]["RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT"])
        
        left_range = f"chr{left_chr} {int(group['LEFT COORDINATE'].min())}-{int(group['LEFT COORDINATE'].max())}"
        right_range = f"chr{right_chr} {int(group['RIGHT COORDINATE'].min())}-{int(group['RIGHT COORDINATE'].max())}"
    
        if group_size > 1:
            summary_row = {
                'GROUP TAG': summary_label,
                'NOTATION': notation,
                'LEFT GENE': left_gene,
                'RIGHT GENE': right_gene,
                'GROUPED TYPE': rearr_type,
                "LEFT SIDE (5') OF NON-INVERTED BREAKPOINT": left_range,
                "RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT": right_range,
                'row_id': '',
                'subgroup': subgroup_id,
                'subgroup_size': group_size,
            }
            
            for col in summary_cols:
                if col in group.columns:
                    if col not in ["NOTATION", "GROUPED TYPE", "LEFT GENE", "RIGHT GENE", "LEFT SIDE (5') OF NON-INVERTED BREAKPOINT", "RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT"]:
                        summary_row[col] = pd.to_numeric(group[col], errors='coerce').sum()
    
            group['GROUP TAG'] = ''
            summary_blocks.append(pd.DataFrame([summary_row]))
            summary_blocks.append(group)
    
            blank_row = pd.DataFrame([{col: '' for col in group.columns}])
            summary_blocks.append(blank_row)
    
        else:
            group['GROUP TAG'] = ''
            ungrouped_blocks.append(group)

    # Add ungrouped rows if present
    if ungrouped_blocks:
        ungrouped = pd.concat(ungrouped_blocks, ignore_index=True)
        ungrouped = ungrouped.sort_values(by='translocation_depth', ascending=False, na_position='last')
    
        # Define GENE_PAIRs of interest
        pairs_of_interest = {
            'MYH11::CBFB', 'CBFB::MYH11',
            'RUNX1::RUNX1T1', 'RUNX1T1::RUNX1',
            'DEK::NUP214', 'NUP214::DEK',
            'KMT2A::MLLT3', 'MLLT3::KMT2A',
            'BCR::ABL1', 'ABL1::BCR',
            'PML::RARA', 'RARA::PML'
        }
    
        # Split ungrouped into two sets
        ungrouped_of_interest = ungrouped[ungrouped['GENE_PAIR'].isin(pairs_of_interest)].copy()
        ungrouped_rest = ungrouped[~ungrouped['GENE_PAIR'].isin(pairs_of_interest)].copy()
    
        # Sort the 'of interest' set by GENE_PAIR then translocation_depth
        ungrouped_of_interest = ungrouped_of_interest.sort_values(
            by=['GENE_PAIR', 'translocation_depth'], ascending=[True, False]
        )
    
        # Add label and rows for "ungrouped, but potentially of interest"
        if not ungrouped_of_interest.empty:
            blank_row = pd.DataFrame([{col: '' for col in group.columns}])
            ungrouped_interest_summary = pd.DataFrame([{
                'GROUP TAG': 'UNGROUPED, BUT POTENTIALLY OF INTEREST',
                **{col: '' for col in ungrouped_of_interest.columns if col != 'GROUP TAG'}
            }])
            summary_blocks.append(ungrouped_interest_summary)
            summary_blocks.append(ungrouped_of_interest)
            summary_blocks.append(blank_row)
    
        # Add label and rows for the remaining ungrouped
        if not ungrouped_rest.empty:
            ungrouped_summary = pd.DataFrame([{
                'GROUP TAG': 'UNGROUPED',
                **{col: '' for col in ungrouped_rest.columns if col != 'GROUP TAG'}
            }])
            summary_blocks.append(ungrouped_summary)
            summary_blocks.append(ungrouped_rest)
    
    # STEP 8: Finalize output
    final_df = pd.concat(summary_blocks, ignore_index=True)

    # # Count number of non-zero and non-empty entries across columns O to BC
    # evidence_cols = final_df.columns[14:55]
    # final_df['evidence_count'] = final_df[evidence_cols].apply(
    #     lambda row: sum(pd.to_numeric(row, errors='coerce').fillna(0) > 0),
    #     axis=1
    # )

    # Determine the dynamic range of evidence columns (those between 'max_VAF' and 'LEFT GENE' in the original input)
    input_df = pd.read_csv(folder+'/'+sample_name+'_translocations_found_just_those_specifically_targeted_both_sides_panel.csv')
    col_list = list(input_df.columns)
    max_vaf_idx = col_list.index('max_VAF')  # should exist after your processing    
    left_gene_idx = col_list.index('LEFT GENE')
    evidence_cols = col_list[max_vaf_idx+1:left_gene_idx]
    
    # Now apply the count across these evidence columns
    evidence_cols = [col for col in evidence_cols if col in final_df.columns]  # ensure all exist
    final_df['evidence_count'] = final_df[evidence_cols].apply(
        lambda row: sum(pd.to_numeric(row, errors='coerce').fillna(0) > 0),
        axis=1
    )
    
    # Make GROUP TAG the first column
    cols = ['GROUP TAG'] + [col for col in final_df.columns if col != 'GROUP TAG']
    final_df = final_df[cols]
    
    # Column order preference
    fixed_columns = ['GROUP TAG', 'NOTATION', "LEFT SIDE (5') OF NON-INVERTED BREAKPOINT", "RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT",
                     'GROUPED TYPE', 'TYPE', 'translocation_depth', 'normal_depth_left', 'normal_depth_right', 'max_normal_depth',
                     'VAF_left', 'VAF_right', 'max_VAF', 'evidence_count', 'discordant reads', 'split reads']
    other_columns = [col for col in final_df.columns if col not in fixed_columns]
    new_order = fixed_columns + other_columns
    final_df = final_df[new_order]
    final_df = final_df.drop(columns=['row_id', 'subgroup', 'SORT_ORDER', 'subgroup_size'], errors='ignore')
    
    final_df.to_csv(folder+'/'+sample_name+'_translocations_found_just_those_specifically_targeted_both_sides_panel_grouped_and_filtered_v3.csv', index=False)
    
    # Save final_df to Excel with formatting
    output_file = folder+'/'+sample_name+'_translocations_found_just_those_specifically_targeted_both_sides_panel_grouped_and_filtered_v3.xlsx'
    
    with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
        final_df.to_excel(writer, index=False, sheet_name='Summary')
    
        workbook  = writer.book
        worksheet = writer.sheets['Summary']
    
        # Freeze the top row
        worksheet.freeze_panes(1, 0)
    
        # Autosize all columns
        for i, col in enumerate(final_df.columns):
            # Find length of longest entry in the column (including header)
            series = final_df[col].astype(str)
            max_len = max(series.map(len).max(), len(str(col)))  # +2 for padding
            worksheet.set_column(i, i, min(max_len, 35))
    
        # Narrower widths for P to BC
        for col_idx in range(15, 55):
            worksheet.set_column(col_idx, col_idx, 8)
    
        # Define formatting styles
        group_summary_format = workbook.add_format({
            'bold': True,
            'bg_color': '#D9E1F2',  # light blue
            'border': 0
        })
        
        ungrouped_summary_format = workbook.add_format({
            'bold': True,
            'bg_color': '#FCE4D6',  # light orange-peach
            'border': 0
        })
        
        # Loop through rows and apply conditional formatting
        for idx, value in enumerate(final_df['GROUP TAG']):
            if pd.notna(value) and value != '':
                if 'UNGROUPED' in value.upper():
                    worksheet.set_row(idx + 1, None, ungrouped_summary_format)  # offset by 1 for header
                else:
                    worksheet.set_row(idx + 1, None, group_summary_format)

    
    return final_df

In [ ]:
parent_folder = Path("UKCTOCS_final_timepoints/CONTROLS")  # Replace with your path
controls = [f.name for f in parent_folder.iterdir() if f.is_dir()]

parent_folder = Path("UKCTOCS_final_timepoints/CASES")  # Replace with your path
cases = [f.name for f in parent_folder.iterdir() if f.is_dir()]

print(controls)
print(cases)

In [ ]:
# Define AML-associated canonical fusions specifically targeted by the panel
targeted_canonical_fusions = {
    'KMT2A::MLLT3', 'BCR::ABL1', 'PML::RARA',
    'RUNX1::RUNX1T1', 'DEK::NUP214', 'CBFB::MYH11', 'MYH11::CBFB'
}
# Define reciprocal fusions that should bypass the read-based filters
reciprocal_fusions = {
    'MLLT3::KMT2A', 'ABL1::BCR', 'RARA::PML',
    'RUNX1T1::RUNX1', 'NUP214::DEK', 'MYH11::CBFB', 'CBFB::MYH11'
}

# Apply read evidence filter, keeping empty rows
def passes_read_evidence_filter(row):
    gene_pair = row.get('GENE_PAIR', None)
    # print(gene_pair)
    
    # Keep empty rows (e.g. inserted spacers)
    if pd.isna(gene_pair) or str(gene_pair).strip() == '':
        return True

    depth = row.get('translocation_depth', 0)
    evidence = row.get('evidence_count', 0)
    discordant = row.get('discordant reads', 0)

    # Always keep reciprocal fusions of targeted events
    if gene_pair in reciprocal_fusions:
        # print('gene_pair in reciprocal_fusions')
        return True

    # If it's a targeted AML-associated rearrangement
    if gene_pair in targeted_canonical_fusions:
        return evidence >= 3 and discordant >= 3

    # print()

    # All other fusions must meet full criteria
    return evidence >= 5 and discordant >= 5

In [ ]:
# # Map each reciprocal to its canonical partner
# reciprocal_to_canonical = {
#     'MLLT3::KMT2A':  'KMT2A::MLLT3',
#     'ABL1::BCR':     'BCR::ABL1',
#     'RARA::PML':     'PML::RARA',
#     'RUNX1T1::RUNX1':'RUNX1::RUNX1T1',
#     'NUP214::DEK':   'DEK::NUP214',
# }

# # Mutual-canonical pair for inv(16)
# mutual_pair = {
#     'CBFB::MYH11': 'MYH11::CBFB',
#     'MYH11::CBFB': 'CBFB::MYH11',
# }

# def _norm(s: pd.Series) -> pd.Series:
#     return s.astype(str).str.strip()

# def _meets_threshold(depth, evidence, discordant):
#     d = pd.to_numeric(depth, errors='coerce').fillna(0)
#     e = pd.to_numeric(evidence, errors='coerce').fillna(0)
#     x = pd.to_numeric(discordant, errors='coerce').fillna(0)
#     return ((d < 50) & (e >= 3) & (x >= 3)) | ((d >= 50) & (e >= 5) & (x >= 5))

# def filter_reciprocals_with_rules(df_all: pd.DataFrame) -> pd.DataFrame:
#     req = {'SAMPLE NAME','GENE_PAIR','translocation_depth','evidence_count','discordant reads'}
#     miss = req - set(df_all.columns)
#     if miss:
#         raise KeyError(f"Missing required columns: {miss}")

#     df = df_all.copy()
#     df['SAMPLE NAME'] = _norm(df['SAMPLE NAME'])
#     df['GENE_PAIR']   = _norm(df['GENE_PAIR'])

#     # Spacers (kept)
#     is_spacer = df['GENE_PAIR'].isna() | (df['GENE_PAIR'] == '') | (df['GENE_PAIR'] == 'nan')

#     # Per-row thresholds
#     df['__passes__'] = _meets_threshold(df['translocation_depth'], df['evidence_count'], df['discordant reads'])

#     gp     = df['GENE_PAIR']
#     sample = df['SAMPLE NAME']

#     # Lookup: does (sample, gene_pair) pass?
#     pass_lookup = df.set_index(['SAMPLE NAME','GENE_PAIR'])['__passes__'].to_dict()

#     # --- Standard reciprocals: keep iff canonical present & passes in same sample ---
#     is_std_recip = gp.isin(RECIP_TO_CANON.keys())
#     std_canon = gp.map(RECIP_TO_CANON)
#     std_canon_passes_same_sample = pd.Series(
#         [pass_lookup.get((s, c), False) if isinstance(c, str) else False
#          for s, c in zip(sample, std_canon)],
#         index=df.index
#     )

#     # --- inv(16) mutual pair: keep if self passes OR partner passes in same sample ---
#     is_mutual = gp.isin(MUTUAL_PAIR.keys())
#     partner = gp.map(MUTUAL_PAIR)
#     partner_passes_same_sample = pd.Series(
#         [pass_lookup.get((s, p), False) if isinstance(p, str) else False
#          for s, p in zip(sample, partner)],
#         index=df.index
#     )
#     keep_mutual = is_mutual & (df['__passes__'] | partner_passes_same_sample)

#     # Non-reciprocals: leave as-is (they’re governed by your other filters)
#     keep_non_recip = ~(is_std_recip | is_mutual)

#     # Final keep mask
#     keep = is_spacer | keep_non_recip | (is_std_recip & std_canon_passes_same_sample) | keep_mutual

#     return df[keep].drop(columns=['__passes__']).copy()

In [ ]:
for sample_name in cases:
    if sample_name not in ['C92_043_s6', 'C92_009_s9', 'C92_062_s3']:
        print(sample_name)
        filter_and_group_chromosomal_rearrangements_v3('UKCTOCS_final_timepoints/CASES/'+sample_name, sample_name)

In [ ]:
for sample_name in controls:
    # if sample_name not in ['CNTRL_197_s6', 'CNTRL_186_s4', 'CNTRL_193_s2', 'CNTRL_203_s5', 'CNTRL_199_s7', 'CNTRL_200_s2', 'CNTRL_204_s5']:
    if sample_name not in ['CNTRL_197_s6']:
        print(sample_name)
        filter_and_group_chromosomal_rearrangements_v3('UKCTOCS_final_timepoints/CONTROLS/'+sample_name, sample_name)

In [ ]:
positive_controls = ['24642_1_xGenUDI28_t911', '24642_25_xGenUDI26_t911', '24642_5_xGenUDI27_t911', '24792_1_xGenUDI32_t69_t315', 
                     '24792_25_xGenUDI30_t69_t315', '24792_50_xGenUDI29_t69_t315', '24792_5_xGenUDI31_t69_t315', '39321_1_xGenUDI40_t821', 
                     '39321_25_xGenUDI38_t821', '39321_50_xGenUDI37_t821', '39321_5_xGenUDI39_t821', 
                     '39326_1_xGenUDI36_inv16', '39326_25_xGenUDI34_inv16', '39326_50_xGenUDI33_inv16', '39326_5_xGenUDI35_inv16', 'LEG16_V12_xGenUDI25']

for positive_control in positive_controls:
    print(positive_control)
    if positive_control not in ['39326_1_xGenUDI36_inv16', 'LEG16_V12_xGenUDI25']:
        filter_and_group_chromosomal_rearrangements_v3(positive_control, positive_control+'_SSCS')

# Combining all the spreadsheets

In [ ]:
def create_combined_grouped_spreadsheet(cases_or_controls, samples):

    # Define your root folder where all sample subfolders are located
    root_dir = Path("UKCTOCS_final_timepoints/"+cases_or_controls)  # <-- EDIT THIS
    
    # Initialise lists to store combined data
    combined_grouped = []
    
    for sample in sorted(samples):
        # print(sample)
        sample_dir = root_dir / sample
        excel_files = list(sample_dir.glob("*translocations_found_just_those_specifically_targeted_both_sides_panel_grouped_and_filtered_v3.xlsx"))
        
        # print(excel_files)
        if not excel_files:
            print(f"⚠️ No Excel file in {sample}")
            continue
            
        excel_file = excel_files[0]
    
        # Load the file
        # print(sample)
        df = pd.read_excel(excel_file)
    
        group_mask = df["GROUP TAG"].astype(str).str.startswith("GROUP:")
        df_grouped = df[group_mask].copy()
        
        # Add a sample column if needed:
        df_grouped["SAMPLE NAME"] = sample
        df_grouped['GENE_PAIR'] = fusion_genes(df_grouped['LEFT GENE'], df_grouped['RIGHT GENE'])
        
        if not df_grouped.empty:
            combined_grouped.append(df_grouped)
            # Add blank row for spacing
            blank_row = pd.Series([None] * len(df_grouped.columns), index=df_grouped.columns)
            combined_grouped.append(pd.DataFrame([blank_row]))
        else:
            print('no grouped events for '+sample)
        
    if combined_grouped:
        # Save final_df to Excel with formatting
        df_all_grouped = pd.concat(combined_grouped, ignore_index=True)
        cols = ["SAMPLE NAME"] + [col for col in df_all_grouped.columns if col != "SAMPLE NAME"]
        df_all_grouped = df_all_grouped[cols]
        # print(df_all_grouped)
        output_file = "UKCTOCS_final_timepoints/Combined_GROUPED_sections_"+cases_or_controls+"_v3.xlsx"

        #remove rearrangements that aren't in the correct orientation
        df_filtered = df_all_grouped[df_all_grouped.apply(matches_expected_inversion, axis=1)].copy()
        #apply read depth and evidence filters
        df_filtered = df_filtered[df_filtered.apply(passes_read_evidence_filter, axis=1)].copy()

        # normalize empties to NA
        df_filtered = df_filtered.applymap(lambda x: x.strip() if isinstance(x, str) else x)
        df_filtered = df_filtered.replace(r'^\s*$', pd.NA, regex=True)
        is_blank = df_filtered.isna().all(axis=1)
        keep = ~is_blank | (is_blank & ~is_blank.shift(fill_value=False))  # keep first of each blank run
        df_filtered = df_filtered[keep]
        # drop leading/trailing blank rows
        if not df_filtered.empty and df_filtered.iloc[0].isna().all():
            df_filtered = df_filtered.iloc[1:]
        if not df_filtered.empty and df_filtered.iloc[-1].isna().all():
            df_filtered = df_filtered.iloc[:-1]
        df_filtered = df_filtered.reset_index(drop=True)
        
        with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
            df_filtered.to_excel(writer, index=False, sheet_name='Summary')
        
            workbook  = writer.book
            worksheet = writer.sheets['Summary']
        
            # Freeze the top row
            worksheet.freeze_panes(1, 0)
        
            # Autosize all columns
            for i, col in enumerate(df_filtered.columns):
                # Find length of longest entry in the column (including header)
                series = df_filtered[col].astype(str)
                max_len = max(series.map(len).max(), len(str(col)))  # +2 for padding
                worksheet.set_column(i, i, min(max_len, 35))
        
            # Narrower widths for P to BC
            for col_idx in range(16, 56):
                worksheet.set_column(col_idx, col_idx, 8)
    
            # Define GENE_PAIRs of interest
            pairs_of_interest = {
                'MYH11::CBFB', 'CBFB::MYH11',
            
                'RUNX1::RUNX1T1', 'RUNX1T1::RUNX1',
                'DEK::NUP214', 'NUP214::DEK',
                'KMT2A::MLLT3', 'MLLT3::KMT2A',
                'BCR::ABL1', 'ABL1::BCR',
                'PML::RARA', 'RARA::PML'
            }
    
            # Define formatting for rows with GENE_PAIRs of interest
            rearrangements_of_interest = workbook.add_format({
                'bg_color': '#D9E1F2'  # light blue
            })
            
            # Apply formatting ONLY to rows with matching GENE_PAIRs
            for idx, gene_pair in enumerate(df_filtered['GENE_PAIR'], start=1):  # Excel data starts on row 2
                if isinstance(gene_pair, str) and gene_pair.strip() in pairs_of_interest:
                    worksheet.set_row(idx, None, rearrangements_of_interest)
    
    return print("✅ Done")

In [ ]:
create_combined_grouped_spreadsheet('CASES', cases)
create_combined_grouped_spreadsheet('CONTROLS', controls)

In [ ]:
def create_combined_ungrouped_but_interest_spreadsheet(cases_or_controls, samples):
    # Define your root folder where all sample subfolders are located
    root_dir = Path("UKCTOCS_final_timepoints/"+cases_or_controls)  # <-- EDIT THIS
    
    # Initialise lists to store combined data
    combined_ungrouped = []
    
    for sample in sorted(samples):
        sample_dir = root_dir / sample
        files = list(sample_dir.glob("*translocations_found_just_those_specifically_targeted_both_sides_panel.csv"))
        
        # print(excel_files)
        if not files:
            print(f"⚠️ No csv file in {sample}")
            continue
            
        csv_file = files[0]
        df = pd.read_csv(csv_file)
        df['LEFT COORDINATE'] = df["LEFT SIDE (5') OF NON-INVERTED BREAKPOINT"].apply(extract_position)
        df['RIGHT COORDINATE'] = df["RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT"].apply(extract_position)
        df = df[df['LEFT GENE'] != df['RIGHT GENE']] #remove rows where the rearrangement supposedly occurs within the same gene
        df['max_normal_depth'] = maximum_of_two_columns(df['normal_depth_left'], df['normal_depth_right'])
        df['max_VAF'] = maximum_of_two_columns(df['VAF_left'], df['VAF_right'])

        # Strip " on left" or " on right" from inversion descriptions
        df['GROUPED TYPE'] = df['TYPE'].astype(str).str.replace(
            r' ON (LEFT|RIGHT)', '', regex=True
        )
            
    
        # Determine the dynamic range of evidence columns (those between 'max_VAF' and 'LEFT GENE' in the original input)
        col_list = list(df.columns)
        max_vaf_idx = col_list.index('max_VAF')  # should exist after your processing    
        left_gene_idx = col_list.index('LEFT GENE')
        evidence_cols = col_list[max_vaf_idx+1:left_gene_idx]
        
        # Now apply the count across these evidence columns
        evidence_cols = [col for col in evidence_cols if col in df.columns]  # ensure all exist
        df['evidence_count'] = df[evidence_cols].apply(
            lambda row: sum(pd.to_numeric(row, errors='coerce').fillna(0) > 0),
            axis=1
        )

        # Define discordant read columns (start with any of: "D_", "'D_", "('D_")
        discordant_cols = [
            col for col in df.columns
            if isinstance(col, str) and (
                col.startswith("D_") or
                col.startswith("'D_") or
                col.startswith("('D_")
            )
        ]
        
        # Define split (concordant) read columns (start with: "C_", "'C_", "('C_", or "('concordant")
        split_cols = [
            col for col in df.columns
            if isinstance(col, str) and (
                col.startswith("C_") or
                col.startswith("'C_") or
                col.startswith("('C_") or
                col.startswith("('concordant")
            )
        ]
        
        # Create the new summed columns
        df["discordant reads"] = df[discordant_cols].sum(axis=1, skipna=True)
        df["split reads"] = df[split_cols].sum(axis=1, skipna=True)

        
        # STEP 1: Reset and label rows
        df = df.reset_index(drop=True)  # start fresh
        df['row_id'] = df.index         # assign a unique integer row ID
        df['GENE_PAIR'] = fusion_genes(df['LEFT GENE'], df['RIGHT GENE'])
        df['GROUP TAG'] = ''
            # Add a sample column if needed:
        df["SAMPLE NAME"] = sample
        
        # STEP 2: Sort
        df = df.sort_values(by=['NOTATION', 'GENE_PAIR', 'LEFT GENE', 'RIGHT GENE', 'GROUPED TYPE', 'LEFT COORDINATE', 'RIGHT COORDINATE'])
        
        # STEP 3: Build groups
        grouped_blocks = []
        group_id_base = 0
        
        for _, group in df.groupby(['NOTATION', 'LEFT GENE', 'RIGHT GENE', 'TYPE']):
            g = group.copy()
            G = nx.Graph()    
            row_ids = g['row_id'].tolist()
            G.add_nodes_from(row_ids)
            g['subgroup'] = np.nan
        
            for i, j in combinations(g.index, 2):
                row_i = g.loc[i]
                row_j = g.loc[j]
                if (
                    abs(row_i['LEFT COORDINATE'] - row_j['LEFT COORDINATE']) <= 500
                    and abs(row_i['RIGHT COORDINATE'] - row_j['RIGHT COORDINATE']) <= 500
                ):
                    G.add_edge(row_i['row_id'], row_j['row_id'])
        
            for group_num, component in enumerate(nx.connected_components(G), start=group_id_base):
                idx = g['row_id'].isin(component)
                g.loc[idx, 'subgroup'] = group_num
        
            group_id_base = int(g['subgroup'].max()) + 1
            grouped_blocks.append(g)
    
        if grouped_blocks:
            # STEP 4: Combine and annotate group sizes
            df_with_groups = pd.concat(grouped_blocks).reset_index(drop=True)
            df_with_groups['subgroup'] = df_with_groups['subgroup'].astype(int)
            group_sizes = df_with_groups['subgroup'].value_counts()
            df_with_groups['subgroup_size'] = df_with_groups['subgroup'].map(group_sizes)
            
            # STEP 5: Rank gene pairs by frequency
            gene_pair_sizes = df_with_groups['GENE_PAIR'].value_counts()
            gene_pair_rank = {gp: i for i, gp in enumerate(gene_pair_sizes.index, start=1)}
            df_with_groups['SORT_ORDER'] = df_with_groups['GENE_PAIR'].map(gene_pair_rank)
            
            summary_cols = [
                'D_NO_OVERLAP',
                ('D_NPP', 'C_SUPP_NPP'),
                ('D_NPP',),
                ('D_NPP', 'softclip_mapping'),
                ('D_SUPP_PP', 'C_PP'),
                ('D_SUPP_NPP', 'D_NPP'),
                ('concordant_1_end_mapping',),
                ('D_NPP_MPP', 'softclip_mapping'),
                ('C_SUPP_NPP', 'softclip_mapping'),
                ('D_SUPP_NPP', 'D_NPP_MPP'),
                ('C_PP',),
                ('D_SUPP_NPP', 'D_NPP', 'C_SUPP_NPP'),
                ('D_SUPP_NPP', 'C_NPP'),
                ('D_SUPP_NPP', 'C_SUPP_NPP'),
                ('D_SUPP_NPP', 'softclip_mapping'),
                ('D_NPP_MPP', 'C_SUPP_NPP'),
                ('C_PP', 'softclip_mapping'),
                ('C_NPP', 'softclip_mapping'),
                ('D_NPP_LEFT_INV', 'C_SUPP_NPP_LEFT'),
                ('D_NPP_LEFT_INV',),
                ('D_NPP_LEFT_INV', 'inv_softclip_mapping'),
                ('D_SUPP_NPP_LEFT', 'D_NPP_LEFT_INV'),
                ('D_SUPP_PP_INV', 'C_PP'),
                'D_NO_OVERLAP_INV',
                ('D_SUPP_NPP_INV', 'D_NPP'),
                ('D_SUPP_NPP_INV', 'D_NPP_LEFT_INV'),
                ('D_SUPP_NPP_INV', 'inv_softclip_mapping'),
                ('C_PP', 'inv_softclip_mapping'),
                ('D_SUPP_PP_INV', 'inv_softclip_mapping'),
                ('D_NPP', 'inv_softclip_mapping'),
                ('D_SUPP_NPP_LEFT', 'inv_softclip_mapping'),
                ('D_NPP_RIGHT_INV', 'C_SUPP_NPP_RIGHT'),
                ('C_SUPP_NPP_RIGHT', 'inv_softclip_mapping'),
                ('D_NPP_RIGHT_INV',),
                ('D_SUPP_NPP_RIGHT', 'D_NPP_RIGHT_INV'),
                ('D_NPP_RIGHT_INV', 'inv_softclip_mapping'),
                ('D_SUPP_NPP_INV', 'D_NPP_RIGHT_INV', 'C_SUPP_NPP_RIGHT'),
                ('D_SUPP_NPP_INV', 'D_NPP_RIGHT_INV'),
                ('C_SUPP_NPP_RIGHT',),
                ('D_SUPP_NPP_RIGHT', 'inv_softclip_mapping'),
                ('translocation_depth'),
                ('normal_depth_left'),
                ('normal_depth_right'),
                ('VAF_left'),
                ('VAF_right'),
                ('max_VAF'),
                ('discordant reads'),
                ('split reads')
            ]
            
            # STEP 6: Final sort for grouping
            df_sorted = df_with_groups.sort_values(
                by=['SORT_ORDER', 'GENE_PAIR', 'subgroup_size', 'subgroup', 'LEFT COORDINATE'],
                ascending=[True, True, False, True, True]
            ).reset_index(drop=True)
        
            # STEP 7: Build summary and output
            combined_ungrouped_blocks = []
            ungrouped_blocks = []
            
            # Define summary columns to sum
            summary_cols = [col for col in df.columns if col not in [
                'GROUP TAG', 'row_id', 'subgroup', 'subgroup_size',
                'LEFT COORDINATE', 'RIGHT COORDINATE', 'FIRST CHROMOSOME', 'SECOND CHROMOSOME',
                'LEFT GENE', 'RIGHT GENE', 'GENE_PAIR', 'GROUPED TYPE', 'NOTATION'
            ]]
            
            for subgroup_id, group in df_sorted.groupby('subgroup', sort=False):
                # group = group.copy()
                group = group.sort_values(by='translocation_depth', ascending=False, na_position='last').copy()
                group_size = len(group)
                # summary_label = f"{group.iloc[0]['NOTATION']} {group.iloc[0]['LEFT GENE']}::{group.iloc[0]['RIGHT GENE']} {group.iloc[0]['TYPE']}"
                summary_label = f"GROUP: {group.iloc[0]['NOTATION']} {group.iloc[0]['LEFT GENE']}::{group.iloc[0]['RIGHT GENE']}"
                notation = group.iloc[0]['NOTATION']
                left_gene = group.iloc[0]['LEFT GENE']
                right_gene = group.iloc[0]['RIGHT GENE']
                rearr_type = group.iloc[0]['GROUPED TYPE']
                gene_pair = group.iloc[0]['GENE_PAIR']
            
                left_chr = extract_chromosome(group.iloc[0]["LEFT SIDE (5') OF NON-INVERTED BREAKPOINT"])
                right_chr = extract_chromosome(group.iloc[0]["RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT"])
                
                left_range = f"chr{left_chr} {int(group['LEFT COORDINATE'].min())}-{int(group['LEFT COORDINATE'].max())}"
                right_range = f"chr{right_chr} {int(group['RIGHT COORDINATE'].min())}-{int(group['RIGHT COORDINATE'].max())}"
            
                if group_size == 1:
                    ungrouped_blocks.append(group)
        
            # Add ungrouped rows if present
            if ungrouped_blocks:
                ungrouped = pd.concat(ungrouped_blocks, ignore_index=True)
                ungrouped = ungrouped.sort_values(by='translocation_depth', ascending=False, na_position='last')
            
                # Define GENE_PAIRs of interest
                pairs_of_interest = {
                    'MYH11::CBFB', 'CBFB::MYH11',
                    'RUNX1::RUNX1T1', 'RUNX1T1::RUNX1',
                    'DEK::NUP214', 'NUP214::DEK',
                    'KMT2A::MLLT3', 'MLLT3::KMT2A',
                    'BCR::ABL1', 'ABL1::BCR',
                    'PML::RARA', 'RARA::PML'
                }
                ungrouped_of_interest = ungrouped[ungrouped['GENE_PAIR'].isin(pairs_of_interest)].copy()
                ungrouped_of_interest = ungrouped_of_interest.sort_values(
                    by=['GENE_PAIR', 'translocation_depth'], ascending=[True, False]
                )
    
                fixed_columns = ['SAMPLE NAME', 'NOTATION', "LEFT SIDE (5') OF NON-INVERTED BREAKPOINT", "RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT",
                                 'GROUPED TYPE', 'TYPE', 'translocation_depth', 'normal_depth_left', 'normal_depth_right', 'max_normal_depth',
                                 'VAF_left', 'VAF_right', 'max_VAF', 'evidence_count', 'discordant reads', 'split reads']
                other_columns = [col for col in ungrouped_of_interest.columns if col not in fixed_columns]
                new_order = fixed_columns + other_columns
                ungrouped_of_interest = ungrouped_of_interest[new_order]
                ungrouped_of_interest = ungrouped_of_interest.drop(columns=['row_id', 'subgroup', 'SORT_ORDER', 'subgroup_size'], errors='ignore')
                
                if not ungrouped_of_interest.empty:
                    blank_row = pd.DataFrame([{col: '' for col in group.columns}])
                    combined_ungrouped.append(ungrouped_of_interest)
                    combined_ungrouped.append(blank_row)
                else:
                    print('no ungrouped events for '+sample)
    
    if combined_ungrouped:
        df_all_ungrouped = pd.concat(combined_ungrouped, ignore_index=True)
        # Save final_df to Excel with formatting
        output_file = "UKCTOCS_final_timepoints/Combined_UNGROUPED_sections_"+cases_or_controls+"_v3.xlsx"

        #remove rearrangements that aren't in the correct orientation
        df_filtered_ungrouped = df_all_ungrouped[df_all_ungrouped.apply(matches_expected_inversion, axis=1)].copy()
        #apply read depth and evidence filters
        df_filtered_ungrouped = df_filtered_ungrouped[df_filtered_ungrouped.apply(passes_read_evidence_filter, axis=1)].copy()

        # normalize empties to NA
        df_filtered_ungrouped = df_filtered_ungrouped.applymap(lambda x: x.strip() if isinstance(x, str) else x)
        df_filtered_ungrouped = df_filtered_ungrouped.replace(r'^\s*$', pd.NA, regex=True)
        is_blank = df_filtered_ungrouped.isna().all(axis=1)
        keep = ~is_blank | (is_blank & ~is_blank.shift(fill_value=False))  # keep first of each blank run
        df_filtered_ungrouped = df_filtered_ungrouped[keep]
        # drop leading/trailing blank rows
        if not df_filtered_ungrouped.empty and df_filtered_ungrouped.iloc[0].isna().all():
            df_filtered_ungrouped = df_filtered_ungrouped.iloc[1:]
        if not df_filtered_ungrouped.empty and df_filtered_ungrouped.iloc[-1].isna().all():
            df_filtered_ungrouped = df_filtered_ungrouped.iloc[:-1]
        df_filtered_ungrouped = df_filtered_ungrouped.reset_index(drop=True)
        
        with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
            df_filtered_ungrouped.to_excel(writer, index=False, sheet_name='Summary')
        
            workbook  = writer.book
            worksheet = writer.sheets['Summary']
        
            # Freeze the top row
            worksheet.freeze_panes(1, 0)
        
            # Autosize all columns
            for i, col in enumerate(df_filtered_ungrouped.columns):
                # Find length of longest entry in the column (including header)
                series = df_filtered_ungrouped[col].astype(str)
                max_len = max(series.map(len).max(), len(str(col)))  # +2 for padding
                worksheet.set_column(i, i, min(max_len, 35))
        
            # Narrower widths for P to BC
            for col_idx in range(16, 56):
                worksheet.set_column(col_idx, col_idx, 8)
    
    return print("✅ Done")

In [ ]:
create_combined_ungrouped_but_interest_spreadsheet('CASES', cases)
create_combined_ungrouped_but_interest_spreadsheet('CONTROLS', controls)

In [ ]:
def create_combined_grouped_spreadsheet_POSITIVE_CONTROLS(samples):

    # Initialise lists to store combined data
    combined_grouped = []
    
    for sample in sorted(samples):
        df = pd.read_excel(sample+'/'+sample+'_translocations_found_just_those_specifically_targeted_both_sides_panel_grouped_and_filtered_v3.xlsx')
    
        group_mask = df["GROUP TAG"].astype(str).str.startswith("GROUP:")
        df_grouped = df[group_mask].copy()
        
        # Add a sample column if needed:
        df_grouped["SAMPLE NAME"] = sample
        df_grouped['GENE_PAIR'] = fusion_genes(df_grouped['LEFT GENE'], df_grouped['RIGHT GENE'])
        
        if not df_grouped.empty:
            combined_grouped.append(df_grouped)
            # Add blank row for spacing
            blank_row = pd.Series([None] * len(df_grouped.columns), index=df_grouped.columns)
            combined_grouped.append(pd.DataFrame([blank_row]))
        else:
            print('no grouped events for '+sample)
        
    if combined_grouped:
        # Save final_df to Excel with formatting
        df_all_grouped = pd.concat(combined_grouped, ignore_index=True)
        cols = ["SAMPLE NAME"] + [col for col in df_all_grouped.columns if col != "SAMPLE NAME"]
        df_all_grouped = df_all_grouped[cols]
        output_file = "Combined_GROUPED_sections_POSITIVE_CONTROLS_v3.xlsx"

        #remove rearrangements that aren't in the correct orientation
        df_filtered = df_all_grouped[df_all_grouped.apply(matches_expected_inversion, axis=1)].copy()
        #apply read depth and evidence filters
        df_filtered = df_filtered[df_filtered.apply(passes_read_evidence_filter, axis=1)].copy()

        # normalize empties to NA
        df_filtered = df_filtered.applymap(lambda x: x.strip() if isinstance(x, str) else x)
        df_filtered = df_filtered.replace(r'^\s*$', pd.NA, regex=True)
        is_blank = df_filtered.isna().all(axis=1)
        keep = ~is_blank | (is_blank & ~is_blank.shift(fill_value=False))  # keep first of each blank run
        df_filtered = df_filtered[keep]
        # drop leading/trailing blank rows
        if not df_filtered.empty and df_filtered.iloc[0].isna().all():
            df_filtered = df_filtered.iloc[1:]
        if not df_filtered.empty and df_filtered.iloc[-1].isna().all():
            df_filtered = df_filtered.iloc[:-1]
        df_filtered = df_filtered.reset_index(drop=True)
        
        with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
            df_filtered.to_excel(writer, index=False, sheet_name='Summary')
        
            workbook  = writer.book
            worksheet = writer.sheets['Summary']
        
            # Freeze the top row
            worksheet.freeze_panes(1, 0)
        
            # Autosize all columns
            for i, col in enumerate(df_filtered.columns):
                # Find length of longest entry in the column (including header)
                series = df_filtered[col].astype(str)
                max_len = max(series.map(len).max(), len(str(col)))  # +2 for padding
                worksheet.set_column(i, i, min(max_len, 35))
        
            # Narrower widths for P to BC
            for col_idx in range(16, 56):
                worksheet.set_column(col_idx, col_idx, 8)
    
            # Define GENE_PAIRs of interest
            pairs_of_interest = {
                'MYH11::CBFB', 'CBFB::MYH11',
                'RUNX1::RUNX1T1', 'RUNX1T1::RUNX1',
                'DEK::NUP214', 'NUP214::DEK',
                'KMT2A::MLLT3', 'MLLT3::KMT2A',
                'BCR::ABL1', 'ABL1::BCR',
                'PML::RARA', 'RARA::PML'
            }
    
            # Define formatting for rows with GENE_PAIRs of interest
            rearrangements_of_interest = workbook.add_format({
                'bg_color': '#D9E1F2'  # light blue
            })
            
            # Apply formatting ONLY to rows with matching GENE_PAIRs
            for idx, gene_pair in enumerate(df_filtered['GENE_PAIR'], start=1):  # Excel data starts on row 2
                if isinstance(gene_pair, str) and gene_pair.strip() in pairs_of_interest:
                    worksheet.set_row(idx, None, rearrangements_of_interest)
    
    return print("✅ Done")

In [ ]:
create_combined_grouped_spreadsheet_POSITIVE_CONTROLS(positive_controls)

In [ ]:
def create_combined_ungrouped_but_interest_spreadsheet_POSITIVE_CONTROLS(samples):

    # Initialise lists to store combined data
    combined_ungrouped = []
    
    for sample in sorted(samples):
        df = pd.read_csv(sample+'/'+sample+'_translocations_found_just_those_specifically_targeted_both_sides_panel.csv')
        df['LEFT COORDINATE'] = df["LEFT SIDE (5') OF NON-INVERTED BREAKPOINT"].apply(extract_position)
        df['RIGHT COORDINATE'] = df["RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT"].apply(extract_position)
        df = df[df['LEFT GENE'] != df['RIGHT GENE']] #remove rows where the rearrangement supposedly occurs within the same gene
        df['max_normal_depth'] = maximum_of_two_columns(df['normal_depth_left'], df['normal_depth_right'])
        df['max_VAF'] = maximum_of_two_columns(df['VAF_left'], df['VAF_right'])
        
        # Strip " on left" or " on right" from inversion descriptions
        df['GROUPED TYPE'] = df['TYPE'].astype(str).str.replace(
            r' ON (LEFT|RIGHT)', '', regex=True
        )
            
    
        # Determine the dynamic range of evidence columns (those between 'max_VAF' and 'LEFT GENE' in the original input)
        col_list = list(df.columns)
        max_vaf_idx = col_list.index('max_VAF')  # should exist after your processing    
        left_gene_idx = col_list.index('LEFT GENE')
        evidence_cols = col_list[max_vaf_idx+1:left_gene_idx]
        
        # Now apply the count across these evidence columns
        evidence_cols = [col for col in evidence_cols if col in df.columns]  # ensure all exist
        df['evidence_count'] = df[evidence_cols].apply(
            lambda row: sum(pd.to_numeric(row, errors='coerce').fillna(0) > 0),
            axis=1
        )

        # Define discordant read columns (start with any of: "D_", "'D_", "('D_")
        discordant_cols = [
            col for col in df.columns
            if isinstance(col, str) and (
                col.startswith("D_") or
                col.startswith("'D_") or
                col.startswith("('D_")
            )
        ]
        
        # Define split (concordant) read columns (start with: "C_", "'C_", "('C_", or "('concordant")
        split_cols = [
            col for col in df.columns
            if isinstance(col, str) and (
                col.startswith("C_") or
                col.startswith("'C_") or
                col.startswith("('C_") or
                col.startswith("('concordant")
            )
        ]
        
        # Create the new summed columns
        df["discordant reads"] = df[discordant_cols].sum(axis=1, skipna=True)
        df["split reads"] = df[split_cols].sum(axis=1, skipna=True)

        
        # STEP 1: Reset and label rows
        df = df.reset_index(drop=True)  # start fresh
        df['row_id'] = df.index         # assign a unique integer row ID
        df['GENE_PAIR'] = fusion_genes(df['LEFT GENE'], df['RIGHT GENE'])
        df['GROUP TAG'] = ''
            # Add a sample column if needed:
        df["SAMPLE NAME"] = sample
        
        # STEP 2: Sort
        df = df.sort_values(by=['NOTATION', 'GENE_PAIR', 'LEFT GENE', 'RIGHT GENE', 'GROUPED TYPE', 'LEFT COORDINATE', 'RIGHT COORDINATE'])
        
        # STEP 3: Build groups
        grouped_blocks = []
        group_id_base = 0
        
        for _, group in df.groupby(['NOTATION', 'LEFT GENE', 'RIGHT GENE', 'GROUPED TYPE']):
            g = group.copy()
            G = nx.Graph()    
            row_ids = g['row_id'].tolist()
            G.add_nodes_from(row_ids)
            g['subgroup'] = np.nan
        
            for i, j in combinations(g.index, 2):
                row_i = g.loc[i]
                row_j = g.loc[j]
                if (
                    abs(row_i['LEFT COORDINATE'] - row_j['LEFT COORDINATE']) <= 500
                    and abs(row_i['RIGHT COORDINATE'] - row_j['RIGHT COORDINATE']) <= 500
                ):
                    G.add_edge(row_i['row_id'], row_j['row_id'])
        
            for group_num, component in enumerate(nx.connected_components(G), start=group_id_base):
                idx = g['row_id'].isin(component)
                g.loc[idx, 'subgroup'] = group_num
        
            group_id_base = int(g['subgroup'].max()) + 1
            grouped_blocks.append(g)
    
        if grouped_blocks:
            # STEP 4: Combine and annotate group sizes
            df_with_groups = pd.concat(grouped_blocks).reset_index(drop=True)
            df_with_groups['subgroup'] = df_with_groups['subgroup'].astype(int)
            group_sizes = df_with_groups['subgroup'].value_counts()
            df_with_groups['subgroup_size'] = df_with_groups['subgroup'].map(group_sizes)
            
            # STEP 5: Rank gene pairs by frequency
            gene_pair_sizes = df_with_groups['GENE_PAIR'].value_counts()
            gene_pair_rank = {gp: i for i, gp in enumerate(gene_pair_sizes.index, start=1)}
            df_with_groups['SORT_ORDER'] = df_with_groups['GENE_PAIR'].map(gene_pair_rank)
            
            summary_cols = [
                'D_NO_OVERLAP',
                ('D_NPP', 'C_SUPP_NPP'),
                ('D_NPP',),
                ('D_NPP', 'softclip_mapping'),
                ('D_SUPP_PP', 'C_PP'),
                ('D_SUPP_NPP', 'D_NPP'),
                ('concordant_1_end_mapping',),
                ('D_NPP_MPP', 'softclip_mapping'),
                ('C_SUPP_NPP', 'softclip_mapping'),
                ('D_SUPP_NPP', 'D_NPP_MPP'),
                ('C_PP',),
                ('D_SUPP_NPP', 'D_NPP', 'C_SUPP_NPP'),
                ('D_SUPP_NPP', 'C_NPP'),
                ('D_SUPP_NPP', 'C_SUPP_NPP'),
                ('D_SUPP_NPP', 'softclip_mapping'),
                ('D_NPP_MPP', 'C_SUPP_NPP'),
                ('C_PP', 'softclip_mapping'),
                ('C_NPP', 'softclip_mapping'),
                ('D_NPP_LEFT_INV', 'C_SUPP_NPP_LEFT'),
                ('D_NPP_LEFT_INV',),
                ('D_NPP_LEFT_INV', 'inv_softclip_mapping'),
                ('D_SUPP_NPP_LEFT', 'D_NPP_LEFT_INV'),
                ('D_SUPP_PP_INV', 'C_PP'),
                'D_NO_OVERLAP_INV',
                ('D_SUPP_NPP_INV', 'D_NPP'),
                ('D_SUPP_NPP_INV', 'D_NPP_LEFT_INV'),
                ('D_SUPP_NPP_INV', 'inv_softclip_mapping'),
                ('C_PP', 'inv_softclip_mapping'),
                ('D_SUPP_PP_INV', 'inv_softclip_mapping'),
                ('D_NPP', 'inv_softclip_mapping'),
                ('D_SUPP_NPP_LEFT', 'inv_softclip_mapping'),
                ('D_NPP_RIGHT_INV', 'C_SUPP_NPP_RIGHT'),
                ('C_SUPP_NPP_RIGHT', 'inv_softclip_mapping'),
                ('D_NPP_RIGHT_INV',),
                ('D_SUPP_NPP_RIGHT', 'D_NPP_RIGHT_INV'),
                ('D_NPP_RIGHT_INV', 'inv_softclip_mapping'),
                ('D_SUPP_NPP_INV', 'D_NPP_RIGHT_INV', 'C_SUPP_NPP_RIGHT'),
                ('D_SUPP_NPP_INV', 'D_NPP_RIGHT_INV'),
                ('C_SUPP_NPP_RIGHT',),
                ('D_SUPP_NPP_RIGHT', 'inv_softclip_mapping'),
                ('translocation_depth'),
                ('normal_depth_left'),
                ('normal_depth_right'),
                ('VAF_left'),
                ('VAF_right'),
                ('max_VAF'),
                ('discordant reads'),
                ('split reads')
            ]
            
            # STEP 6: Final sort for grouping
            df_sorted = df_with_groups.sort_values(
                by=['SORT_ORDER', 'GENE_PAIR', 'subgroup_size', 'subgroup', 'LEFT COORDINATE'],
                ascending=[True, True, False, True, True]
            ).reset_index(drop=True)
        
            # STEP 7: Build summary and output
            combined_ungrouped_blocks = []
            ungrouped_blocks = []
            
            # Define summary columns to sum
            summary_cols = [col for col in df.columns if col not in [
                'GROUP TAG', 'row_id', 'subgroup', 'subgroup_size',
                'LEFT COORDINATE', 'RIGHT COORDINATE', 'FIRST CHROMOSOME', 'SECOND CHROMOSOME',
                'LEFT GENE', 'RIGHT GENE', 'GENE_PAIR', 'GROUPED TYPE', 'NOTATION'
            ]]
            
            for subgroup_id, group in df_sorted.groupby('subgroup', sort=False):
                # group = group.copy()
                group = group.sort_values(by='translocation_depth', ascending=False, na_position='last').copy()
                group_size = len(group)
                # summary_label = f"{group.iloc[0]['NOTATION']} {group.iloc[0]['LEFT GENE']}::{group.iloc[0]['RIGHT GENE']} {group.iloc[0]['TYPE']}"
                summary_label = f"GROUP: {group.iloc[0]['NOTATION']} {group.iloc[0]['LEFT GENE']}::{group.iloc[0]['RIGHT GENE']}"
                notation = group.iloc[0]['NOTATION']
                left_gene = group.iloc[0]['LEFT GENE']
                right_gene = group.iloc[0]['RIGHT GENE']
                rearr_type = group.iloc[0]['GROUPED TYPE']
                gene_pair = group.iloc[0]['GENE_PAIR']
            
                left_chr = extract_chromosome(group.iloc[0]["LEFT SIDE (5') OF NON-INVERTED BREAKPOINT"])
                right_chr = extract_chromosome(group.iloc[0]["RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT"])
                
                left_range = f"chr{left_chr} {int(group['LEFT COORDINATE'].min())}-{int(group['LEFT COORDINATE'].max())}"
                right_range = f"chr{right_chr} {int(group['RIGHT COORDINATE'].min())}-{int(group['RIGHT COORDINATE'].max())}"
            
                if group_size == 1:
                    ungrouped_blocks.append(group)
        
            # Add ungrouped rows if present
            if ungrouped_blocks:
                ungrouped = pd.concat(ungrouped_blocks, ignore_index=True)
                ungrouped = ungrouped.sort_values(by='translocation_depth', ascending=False, na_position='last')
            
                # Define GENE_PAIRs of interest
                pairs_of_interest = {
                    'MYH11::CBFB', 'CBFB::MYH11',
                    'RUNX1::RUNX1T1', 'RUNX1T1::RUNX1',
                    'DEK::NUP214', 'NUP214::DEK',
                    'KMT2A::MLLT3', 'MLLT3::KMT2A',
                    'BCR::ABL1', 'ABL1::BCR',
                    'PML::RARA', 'RARA::PML'
                }
                ungrouped_of_interest = ungrouped[ungrouped['GENE_PAIR'].isin(pairs_of_interest)].copy()
                ungrouped_of_interest = ungrouped_of_interest.sort_values(
                    by=['GENE_PAIR', 'translocation_depth'], ascending=[True, False]
                )
    
                fixed_columns = ['SAMPLE NAME', 'NOTATION', "LEFT SIDE (5') OF NON-INVERTED BREAKPOINT", "RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT",
                                 'GROUPED TYPE', 'TYPE', 'translocation_depth', 'normal_depth_left', 'normal_depth_right', 'max_normal_depth',
                                 'VAF_left', 'VAF_right', 'max_VAF', 'evidence_count', 'discordant reads', 'split reads']
                other_columns = [col for col in ungrouped_of_interest.columns if col not in fixed_columns]
                new_order = fixed_columns + other_columns
                ungrouped_of_interest = ungrouped_of_interest[new_order]
                ungrouped_of_interest = ungrouped_of_interest.drop(columns=['row_id', 'subgroup', 'SORT_ORDER', 'subgroup_size'], errors='ignore')
                
                if not ungrouped_of_interest.empty:
                    blank_row = pd.DataFrame([{col: '' for col in group.columns}])
                    combined_ungrouped.append(ungrouped_of_interest)
                    combined_ungrouped.append(blank_row)
                else:
                    print('no ungrouped events for '+sample)
    
    # if combined_ungrouped:
    #     df_all_ungrouped = pd.concat(combined_ungrouped, ignore_index=True)
    #     cols = ["SAMPLE NAME"] + [col for col in df_all_ungrouped.columns if col != "SAMPLE NAME"]
    #     df_all_ungrouped = df_all_ungrouped[cols]
    #     # df_all_ungrouped_sorted = df_all_ungrouped.sort_values(by=["SAMPLE NAME", "translocation_depth"], ascending=[True, False], na_position="last")
    #     df_all_ungrouped.to_excel(root_dir / "Combined_UNGROUPED_unmatched_sections.xlsx", index=False)
    
    if combined_ungrouped:
        df_all_ungrouped = pd.concat(combined_ungrouped, ignore_index=True)
        # Save final_df to Excel with formatting
        output_file = "Combined_UNGROUPED_sections_POSITIVE_CONTROLS_v3.xlsx"

        #remove rearrangements that aren't in the correct orientation
        df_filtered_ungrouped = df_all_ungrouped[df_all_ungrouped.apply(matches_expected_inversion, axis=1)].copy()
        #apply read depth and evidence filters
        df_filtered_ungrouped = df_filtered_ungrouped[df_filtered_ungrouped.apply(passes_read_evidence_filter, axis=1)].copy()

        # normalize empties to NA
        df_filtered_ungrouped = df_filtered_ungrouped.applymap(lambda x: x.strip() if isinstance(x, str) else x)
        df_filtered_ungrouped = df_filtered_ungrouped.replace(r'^\s*$', pd.NA, regex=True)
        is_blank = df_filtered_ungrouped.isna().all(axis=1)
        keep = ~is_blank | (is_blank & ~is_blank.shift(fill_value=False))  # keep first of each blank run
        df_filtered_ungrouped = df_filtered_ungrouped[keep]
        # drop leading/trailing blank rows
        if not df_filtered_ungrouped.empty and df_filtered_ungrouped.iloc[0].isna().all():
            df_filtered_ungrouped = df_filtered_ungrouped.iloc[1:]
        if not df_filtered_ungrouped.empty and df_filtered_ungrouped.iloc[-1].isna().all():
            df_filtered_ungrouped = df_filtered_ungrouped.iloc[:-1]
        df_filtered_ungrouped = df_filtered_ungrouped.reset_index(drop=True)
        
        with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
            df_filtered_ungrouped.to_excel(writer, index=False, sheet_name='Summary')
        
            workbook  = writer.book
            worksheet = writer.sheets['Summary']
        
            # Freeze the top row
            worksheet.freeze_panes(1, 0)
        
            # Autosize all columns
            for i, col in enumerate(df_filtered_ungrouped.columns):
                # Find length of longest entry in the column (including header)
                series = df_filtered_ungrouped[col].astype(str)
                max_len = max(series.map(len).max(), len(str(col)))  # +2 for padding
                worksheet.set_column(i, i, min(max_len, 35))
        
            # Narrower widths for P to BC
            for col_idx in range(16, 56):
                worksheet.set_column(col_idx, col_idx, 8)
    
    return print("✅ Done")

In [ ]:
create_combined_ungrouped_but_interest_spreadsheet_POSITIVE_CONTROLS(positive_controls)

In [ ]:
def create_combined_ungrouped_not_interesting_spreadsheet_POSITIVE_CONTROLS(samples):

    # Initialise lists to store combined data
    combined_ungrouped = []
    
    for sample in sorted(samples):
        df = pd.read_csv(sample+'/'+sample+'_translocations_found_just_those_specifically_targeted_both_sides_panel.csv')
        df['LEFT COORDINATE'] = df["LEFT SIDE (5') OF NON-INVERTED BREAKPOINT"].apply(extract_position)
        df['RIGHT COORDINATE'] = df["RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT"].apply(extract_position)
        df = df[df['LEFT GENE'] != df['RIGHT GENE']] #remove rows where the rearrangement supposedly occurs within the same gene
        df['max_normal_depth'] = maximum_of_two_columns(df['normal_depth_left'], df['normal_depth_right'])
        df['max_VAF'] = maximum_of_two_columns(df['VAF_left'], df['VAF_right'])
        
        # Strip " on left" or " on right" from inversion descriptions
        df['GROUPED TYPE'] = df['TYPE'].astype(str).str.replace(
            r' ON (LEFT|RIGHT)', '', regex=True
        )
            
        # Determine the dynamic range of evidence columns (those between 'max_VAF' and 'LEFT GENE' in the original input)
        col_list = list(df.columns)
        max_vaf_idx = col_list.index('max_VAF')  # should exist after your processing    
        left_gene_idx = col_list.index('LEFT GENE')
        evidence_cols = col_list[max_vaf_idx+1:left_gene_idx]
        
        # Now apply the count across these evidence columns
        evidence_cols = [col for col in evidence_cols if col in df.columns]  # ensure all exist
        df['evidence_count'] = df[evidence_cols].apply(
            lambda row: sum(pd.to_numeric(row, errors='coerce').fillna(0) > 0),
            axis=1
        )

        # Define discordant read columns (start with any of: "D_", "'D_", "('D_")
        discordant_cols = [
            col for col in df.columns
            if isinstance(col, str) and (
                col.startswith("D_") or
                col.startswith("'D_") or
                col.startswith("('D_")
            )
        ]
        
        # Define split (concordant) read columns (start with: "C_", "'C_", "('C_", or "('concordant")
        split_cols = [
            col for col in df.columns
            if isinstance(col, str) and (
                col.startswith("C_") or
                col.startswith("'C_") or
                col.startswith("('C_") or
                col.startswith("('concordant")
            )
        ]
        
        # Create the new summed columns
        df["discordant reads"] = df[discordant_cols].sum(axis=1, skipna=True)
        df["split reads"] = df[split_cols].sum(axis=1, skipna=True)

        
        # STEP 1: Reset and label rows
        df = df.reset_index(drop=True)  # start fresh
        df['row_id'] = df.index         # assign a unique integer row ID
        df['GENE_PAIR'] = fusion_genes(df['LEFT GENE'], df['RIGHT GENE'])
        df['GROUP TAG'] = ''
            # Add a sample column if needed:
        df["SAMPLE NAME"] = sample
        
        # STEP 2: Sort
        df = df.sort_values(by=['NOTATION', 'GENE_PAIR', 'LEFT GENE', 'RIGHT GENE', 'GROUPED TYPE', 'LEFT COORDINATE', 'RIGHT COORDINATE'])
        
        # STEP 3: Build groups
        grouped_blocks = []
        group_id_base = 0
        
        for _, group in df.groupby(['NOTATION', 'LEFT GENE', 'RIGHT GENE', 'GROUPED TYPE']):
            g = group.copy()
            G = nx.Graph()    
            row_ids = g['row_id'].tolist()
            G.add_nodes_from(row_ids)
            g['subgroup'] = np.nan
        
            for i, j in combinations(g.index, 2):
                row_i = g.loc[i]
                row_j = g.loc[j]
                if (
                    abs(row_i['LEFT COORDINATE'] - row_j['LEFT COORDINATE']) <= 500
                    and abs(row_i['RIGHT COORDINATE'] - row_j['RIGHT COORDINATE']) <= 500
                ):
                    G.add_edge(row_i['row_id'], row_j['row_id'])
        
            for group_num, component in enumerate(nx.connected_components(G), start=group_id_base):
                idx = g['row_id'].isin(component)
                g.loc[idx, 'subgroup'] = group_num
        
            group_id_base = int(g['subgroup'].max()) + 1
            grouped_blocks.append(g)
    
        if grouped_blocks:
            # STEP 4: Combine and annotate group sizes
            df_with_groups = pd.concat(grouped_blocks).reset_index(drop=True)
            df_with_groups['subgroup'] = df_with_groups['subgroup'].astype(int)
            group_sizes = df_with_groups['subgroup'].value_counts()
            df_with_groups['subgroup_size'] = df_with_groups['subgroup'].map(group_sizes)
            
            # STEP 5: Rank gene pairs by frequency
            gene_pair_sizes = df_with_groups['GENE_PAIR'].value_counts()
            gene_pair_rank = {gp: i for i, gp in enumerate(gene_pair_sizes.index, start=1)}
            df_with_groups['SORT_ORDER'] = df_with_groups['GENE_PAIR'].map(gene_pair_rank)
            
            summary_cols = [
                'D_NO_OVERLAP',
                ('D_NPP', 'C_SUPP_NPP'),
                ('D_NPP',),
                ('D_NPP', 'softclip_mapping'),
                ('D_SUPP_PP', 'C_PP'),
                ('D_SUPP_NPP', 'D_NPP'),
                ('concordant_1_end_mapping',),
                ('D_NPP_MPP', 'softclip_mapping'),
                ('C_SUPP_NPP', 'softclip_mapping'),
                ('D_SUPP_NPP', 'D_NPP_MPP'),
                ('C_PP',),
                ('D_SUPP_NPP', 'D_NPP', 'C_SUPP_NPP'),
                ('D_SUPP_NPP', 'C_NPP'),
                ('D_SUPP_NPP', 'C_SUPP_NPP'),
                ('D_SUPP_NPP', 'softclip_mapping'),
                ('D_NPP_MPP', 'C_SUPP_NPP'),
                ('C_PP', 'softclip_mapping'),
                ('C_NPP', 'softclip_mapping'),
                ('D_NPP_LEFT_INV', 'C_SUPP_NPP_LEFT'),
                ('D_NPP_LEFT_INV',),
                ('D_NPP_LEFT_INV', 'inv_softclip_mapping'),
                ('D_SUPP_NPP_LEFT', 'D_NPP_LEFT_INV'),
                ('D_SUPP_PP_INV', 'C_PP'),
                'D_NO_OVERLAP_INV',
                ('D_SUPP_NPP_INV', 'D_NPP'),
                ('D_SUPP_NPP_INV', 'D_NPP_LEFT_INV'),
                ('D_SUPP_NPP_INV', 'inv_softclip_mapping'),
                ('C_PP', 'inv_softclip_mapping'),
                ('D_SUPP_PP_INV', 'inv_softclip_mapping'),
                ('D_NPP', 'inv_softclip_mapping'),
                ('D_SUPP_NPP_LEFT', 'inv_softclip_mapping'),
                ('D_NPP_RIGHT_INV', 'C_SUPP_NPP_RIGHT'),
                ('C_SUPP_NPP_RIGHT', 'inv_softclip_mapping'),
                ('D_NPP_RIGHT_INV',),
                ('D_SUPP_NPP_RIGHT', 'D_NPP_RIGHT_INV'),
                ('D_NPP_RIGHT_INV', 'inv_softclip_mapping'),
                ('D_SUPP_NPP_INV', 'D_NPP_RIGHT_INV', 'C_SUPP_NPP_RIGHT'),
                ('D_SUPP_NPP_INV', 'D_NPP_RIGHT_INV'),
                ('C_SUPP_NPP_RIGHT',),
                ('D_SUPP_NPP_RIGHT', 'inv_softclip_mapping'),
                ('translocation_depth'),
                ('normal_depth_left'),
                ('normal_depth_right'),
                ('VAF_left'),
                ('VAF_right'),
                ('max_VAF'),
                ('discordant reads'),
                ('split reads')
            ]
            
            # STEP 6: Final sort for grouping
            df_sorted = df_with_groups.sort_values(
                by=['SORT_ORDER', 'GENE_PAIR', 'subgroup_size', 'subgroup', 'LEFT COORDINATE'],
                ascending=[True, True, False, True, True]
            ).reset_index(drop=True)
        
            # STEP 7: Build summary and output
            combined_ungrouped_blocks = []
            ungrouped_blocks = []
            
            # Define summary columns to sum
            summary_cols = [col for col in df.columns if col not in [
                'GROUP TAG', 'row_id', 'subgroup', 'subgroup_size',
                'LEFT COORDINATE', 'RIGHT COORDINATE', 'FIRST CHROMOSOME', 'SECOND CHROMOSOME',
                'LEFT GENE', 'RIGHT GENE', 'GENE_PAIR', 'GROUPED TYPE', 'NOTATION'
            ]]
            
            for subgroup_id, group in df_sorted.groupby('subgroup', sort=False):
                # group = group.copy()
                group = group.sort_values(by='translocation_depth', ascending=False, na_position='last').copy()
                group_size = len(group)
                # summary_label = f"{group.iloc[0]['NOTATION']} {group.iloc[0]['LEFT GENE']}::{group.iloc[0]['RIGHT GENE']} {group.iloc[0]['TYPE']}"
                summary_label = f"GROUP: {group.iloc[0]['NOTATION']} {group.iloc[0]['LEFT GENE']}::{group.iloc[0]['RIGHT GENE']}"
                notation = group.iloc[0]['NOTATION']
                left_gene = group.iloc[0]['LEFT GENE']
                right_gene = group.iloc[0]['RIGHT GENE']
                rearr_type = group.iloc[0]['GROUPED TYPE']
                gene_pair = group.iloc[0]['GENE_PAIR']
            
                left_chr = extract_chromosome(group.iloc[0]["LEFT SIDE (5') OF NON-INVERTED BREAKPOINT"])
                right_chr = extract_chromosome(group.iloc[0]["RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT"])
                
                left_range = f"chr{left_chr} {int(group['LEFT COORDINATE'].min())}-{int(group['LEFT COORDINATE'].max())}"
                right_range = f"chr{right_chr} {int(group['RIGHT COORDINATE'].min())}-{int(group['RIGHT COORDINATE'].max())}"
            
                if group_size == 1:
                    ungrouped_blocks.append(group)
        
            # Add ungrouped rows if present
            if ungrouped_blocks:
                ungrouped = pd.concat(ungrouped_blocks, ignore_index=True)
                ungrouped = ungrouped.sort_values(by='translocation_depth', ascending=False, na_position='last')
            
                # Define GENE_PAIRs of interest
                pairs_of_interest = {
                    'MYH11::CBFB', 'CBFB::MYH11',
                    'RUNX1::RUNX1T1', 'RUNX1T1::RUNX1',
                    'DEK::NUP214', 'NUP214::DEK',
                    'KMT2A::MLLT3', 'MLLT3::KMT2A',
                    'BCR::ABL1', 'ABL1::BCR',
                    'PML::RARA', 'RARA::PML'
                }
                ungrouped_not_of_interest = ungrouped[~ungrouped['GENE_PAIR'].isin(pairs_of_interest)].copy() #i.e. not in list of interesting gene pairs
                ungrouped_not_of_interest = ungrouped_not_of_interest.sort_values(
                    by=['GENE_PAIR', 'translocation_depth'], ascending=[True, False]
                )
    
                fixed_columns = ['SAMPLE NAME', 'NOTATION', "LEFT SIDE (5') OF NON-INVERTED BREAKPOINT", "RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT",
                                 'GROUPED TYPE', 'TYPE', 'translocation_depth', 'normal_depth_left', 'normal_depth_right', 'max_normal_depth',
                                 'VAF_left', 'VAF_right', 'max_VAF', 'evidence_count', 'discordant reads', 'split reads']
                other_columns = [col for col in ungrouped_not_of_interest.columns if col not in fixed_columns]
                new_order = fixed_columns + other_columns
                ungrouped_not_of_interest = ungrouped_not_of_interest[new_order]
                ungrouped_not_of_interest = ungrouped_not_of_interest.drop(columns=['row_id', 'subgroup', 'SORT_ORDER', 'subgroup_size'], errors='ignore')
                
                if not ungrouped_not_of_interest.empty:
                    # blank_row = pd.DataFrame([{col: '' for col in group.columns}])
                    combined_ungrouped.append(ungrouped_not_of_interest)
                    # combined_ungrouped.append(blank_row)
                else:
                    print('no ungrouped not of interest events for '+sample)
    
    if combined_ungrouped:
        df_all_ungrouped = pd.concat(combined_ungrouped, ignore_index=True)
        df_all_ungrouped = df_all_ungrouped.replace("", pd.NA).dropna(how='all')
        # Save final_df to Excel with formatting
        output_file = "Combined_UNGROUPED_NOT_OF_INTEREST_sections_POSITIVE_CONTROLS_v3.xlsx"

        #remove rearrangements that aren't in the correct orientation
        df_filtered_ungrouped = df_all_ungrouped[df_all_ungrouped.apply(matches_expected_inversion, axis=1)].copy()
        #apply read depth and evidence filters
        df_filtered_ungrouped = df_filtered_ungrouped[df_filtered_ungrouped.apply(passes_read_evidence_filter, axis=1)].copy()

        # normalize empties to NA
        df_filtered_ungrouped = df_filtered_ungrouped.applymap(lambda x: x.strip() if isinstance(x, str) else x)
        df_filtered_ungrouped = df_filtered_ungrouped.replace(r'^\s*$', pd.NA, regex=True)
        is_blank = df_filtered_ungrouped.isna().all(axis=1)
        keep = ~is_blank | (is_blank & ~is_blank.shift(fill_value=False))  # keep first of each blank run
        df_filtered_ungrouped = df_filtered_ungrouped[keep]
        # drop leading/trailing blank rows
        if not df_filtered_ungrouped.empty and df_filtered_ungrouped.iloc[0].isna().all():
            df_filtered_ungrouped = df_filtered_ungrouped.iloc[1:]
        if not df_filtered_ungrouped.empty and df_filtered_ungrouped.iloc[-1].isna().all():
            df_filtered_ungrouped = df_filtered_ungrouped.iloc[:-1]
        df_filtered_ungrouped = df_filtered_ungrouped.reset_index(drop=True)
        
        with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
            df_filtered_ungrouped.to_excel(writer, index=False, sheet_name='Summary')
        
            workbook  = writer.book
            worksheet = writer.sheets['Summary']
        
            # Freeze the top row
            worksheet.freeze_panes(1, 0)

            ncols = len(df_filtered_ungrouped.columns)
            
            # Explicitly unhide all columns first (just in case)
            worksheet.set_column(0, ncols - 1, None, None, {'hidden': False})
            
            # Sensible default width so nothing starts at 0
            worksheet.set_column(0, ncols - 1, 10)
        
            # Autosize with a minimum width guard
            for i, col in enumerate(df_filtered_ungrouped.columns):
                series = df_filtered_ungrouped[col].astype(str)
                max_len = max(series.map(len).max(), len(str(col)))
                width = max(8, min(max_len, 35))  # enforce min width of 8 chars
                worksheet.set_column(i, i, width)
                    
            # Narrower widths for P to BC
            for col_idx in range(16, 56):
                worksheet.set_column(col_idx, col_idx, 8)
    
    return print("✅ Done")

In [ ]:
create_combined_ungrouped_not_interesting_spreadsheet_POSITIVE_CONTROLS(positive_controls)

In [ ]:
def create_combined_ungrouped_not_interesting_spreadsheet(cases_or_controls, samples):

    root_dir = Path("UKCTOCS_final_timepoints/"+cases_or_controls)  # <-- EDIT THIS
    
    # Initialise lists to store combined data
    combined_ungrouped = []

    for sample in sorted(samples):
        sample_dir = root_dir / sample
        files = list(sample_dir.glob("*translocations_found_just_those_specifically_targeted_both_sides_panel.csv"))
        
        # print(excel_files)
        if not files:
            print(f"⚠️ No csv file in {sample}")
            continue
            
        csv_file = files[0]
        df = pd.read_csv(csv_file)
        df['LEFT COORDINATE'] = df["LEFT SIDE (5') OF NON-INVERTED BREAKPOINT"].apply(extract_position)
        df['RIGHT COORDINATE'] = df["RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT"].apply(extract_position)
        df = df[df['LEFT GENE'] != df['RIGHT GENE']] #remove rows where the rearrangement supposedly occurs within the same gene
        df['max_normal_depth'] = maximum_of_two_columns(df['normal_depth_left'], df['normal_depth_right'])
        df['max_VAF'] = maximum_of_two_columns(df['VAF_left'], df['VAF_right'])
        
        # Strip " on left" or " on right" from inversion descriptions
        df['GROUPED TYPE'] = df['TYPE'].astype(str).str.replace(
            r' ON (LEFT|RIGHT)', '', regex=True
        )
            
        # Determine the dynamic range of evidence columns (those between 'max_VAF' and 'LEFT GENE' in the original input)
        col_list = list(df.columns)
        max_vaf_idx = col_list.index('max_VAF')  # should exist after your processing    
        left_gene_idx = col_list.index('LEFT GENE')
        evidence_cols = col_list[max_vaf_idx+1:left_gene_idx]
        
        # Now apply the count across these evidence columns
        evidence_cols = [col for col in evidence_cols if col in df.columns]  # ensure all exist
        df['evidence_count'] = df[evidence_cols].apply(
            lambda row: sum(pd.to_numeric(row, errors='coerce').fillna(0) > 0),
            axis=1
        )

        # Define discordant read columns (start with any of: "D_", "'D_", "('D_")
        discordant_cols = [
            col for col in df.columns
            if isinstance(col, str) and (
                col.startswith("D_") or
                col.startswith("'D_") or
                col.startswith("('D_")
            )
        ]
        
        # Define split (concordant) read columns (start with: "C_", "'C_", "('C_", or "('concordant")
        split_cols = [
            col for col in df.columns
            if isinstance(col, str) and (
                col.startswith("C_") or
                col.startswith("'C_") or
                col.startswith("('C_") or
                col.startswith("('concordant")
            )
        ]
        
        # Create the new summed columns
        df["discordant reads"] = df[discordant_cols].sum(axis=1, skipna=True)
        df["split reads"] = df[split_cols].sum(axis=1, skipna=True)

        
        # STEP 1: Reset and label rows
        df = df.reset_index(drop=True)  # start fresh
        df['row_id'] = df.index         # assign a unique integer row ID
        df['GENE_PAIR'] = fusion_genes(df['LEFT GENE'], df['RIGHT GENE'])
        df['GROUP TAG'] = ''
            # Add a sample column if needed:
        df["SAMPLE NAME"] = sample
        
        # STEP 2: Sort
        df = df.sort_values(by=['NOTATION', 'GENE_PAIR', 'LEFT GENE', 'RIGHT GENE', 'GROUPED TYPE', 'LEFT COORDINATE', 'RIGHT COORDINATE'])
        
        # STEP 3: Build groups
        grouped_blocks = []
        group_id_base = 0
        
        for _, group in df.groupby(['NOTATION', 'LEFT GENE', 'RIGHT GENE', 'GROUPED TYPE']):
            g = group.copy()
            G = nx.Graph()    
            row_ids = g['row_id'].tolist()
            G.add_nodes_from(row_ids)
            g['subgroup'] = np.nan
        
            for i, j in combinations(g.index, 2):
                row_i = g.loc[i]
                row_j = g.loc[j]
                if (
                    abs(row_i['LEFT COORDINATE'] - row_j['LEFT COORDINATE']) <= 500
                    and abs(row_i['RIGHT COORDINATE'] - row_j['RIGHT COORDINATE']) <= 500
                ):
                    G.add_edge(row_i['row_id'], row_j['row_id'])
        
            for group_num, component in enumerate(nx.connected_components(G), start=group_id_base):
                idx = g['row_id'].isin(component)
                g.loc[idx, 'subgroup'] = group_num
        
            group_id_base = int(g['subgroup'].max()) + 1
            grouped_blocks.append(g)
    
        if grouped_blocks:
            # STEP 4: Combine and annotate group sizes
            df_with_groups = pd.concat(grouped_blocks).reset_index(drop=True)
            df_with_groups['subgroup'] = df_with_groups['subgroup'].astype(int)
            group_sizes = df_with_groups['subgroup'].value_counts()
            df_with_groups['subgroup_size'] = df_with_groups['subgroup'].map(group_sizes)
            
            # STEP 5: Rank gene pairs by frequency
            gene_pair_sizes = df_with_groups['GENE_PAIR'].value_counts()
            gene_pair_rank = {gp: i for i, gp in enumerate(gene_pair_sizes.index, start=1)}
            df_with_groups['SORT_ORDER'] = df_with_groups['GENE_PAIR'].map(gene_pair_rank)
            
            summary_cols = [
                'D_NO_OVERLAP',
                ('D_NPP', 'C_SUPP_NPP'),
                ('D_NPP',),
                ('D_NPP', 'softclip_mapping'),
                ('D_SUPP_PP', 'C_PP'),
                ('D_SUPP_NPP', 'D_NPP'),
                ('concordant_1_end_mapping',),
                ('D_NPP_MPP', 'softclip_mapping'),
                ('C_SUPP_NPP', 'softclip_mapping'),
                ('D_SUPP_NPP', 'D_NPP_MPP'),
                ('C_PP',),
                ('D_SUPP_NPP', 'D_NPP', 'C_SUPP_NPP'),
                ('D_SUPP_NPP', 'C_NPP'),
                ('D_SUPP_NPP', 'C_SUPP_NPP'),
                ('D_SUPP_NPP', 'softclip_mapping'),
                ('D_NPP_MPP', 'C_SUPP_NPP'),
                ('C_PP', 'softclip_mapping'),
                ('C_NPP', 'softclip_mapping'),
                ('D_NPP_LEFT_INV', 'C_SUPP_NPP_LEFT'),
                ('D_NPP_LEFT_INV',),
                ('D_NPP_LEFT_INV', 'inv_softclip_mapping'),
                ('D_SUPP_NPP_LEFT', 'D_NPP_LEFT_INV'),
                ('D_SUPP_PP_INV', 'C_PP'),
                'D_NO_OVERLAP_INV',
                ('D_SUPP_NPP_INV', 'D_NPP'),
                ('D_SUPP_NPP_INV', 'D_NPP_LEFT_INV'),
                ('D_SUPP_NPP_INV', 'inv_softclip_mapping'),
                ('C_PP', 'inv_softclip_mapping'),
                ('D_SUPP_PP_INV', 'inv_softclip_mapping'),
                ('D_NPP', 'inv_softclip_mapping'),
                ('D_SUPP_NPP_LEFT', 'inv_softclip_mapping'),
                ('D_NPP_RIGHT_INV', 'C_SUPP_NPP_RIGHT'),
                ('C_SUPP_NPP_RIGHT', 'inv_softclip_mapping'),
                ('D_NPP_RIGHT_INV',),
                ('D_SUPP_NPP_RIGHT', 'D_NPP_RIGHT_INV'),
                ('D_NPP_RIGHT_INV', 'inv_softclip_mapping'),
                ('D_SUPP_NPP_INV', 'D_NPP_RIGHT_INV', 'C_SUPP_NPP_RIGHT'),
                ('D_SUPP_NPP_INV', 'D_NPP_RIGHT_INV'),
                ('C_SUPP_NPP_RIGHT',),
                ('D_SUPP_NPP_RIGHT', 'inv_softclip_mapping'),
                ('translocation_depth'),
                ('normal_depth_left'),
                ('normal_depth_right'),
                ('VAF_left'),
                ('VAF_right'),
                ('max_VAF'),
                ('discordant reads'),
                ('split reads')
            ]
            
            # STEP 6: Final sort for grouping
            df_sorted = df_with_groups.sort_values(
                by=['SORT_ORDER', 'GENE_PAIR', 'subgroup_size', 'subgroup', 'LEFT COORDINATE'],
                ascending=[True, True, False, True, True]
            ).reset_index(drop=True)
        
            # STEP 7: Build summary and output
            combined_ungrouped_blocks = []
            ungrouped_blocks = []
            
            # Define summary columns to sum
            summary_cols = [col for col in df.columns if col not in [
                'GROUP TAG', 'row_id', 'subgroup', 'subgroup_size',
                'LEFT COORDINATE', 'RIGHT COORDINATE', 'FIRST CHROMOSOME', 'SECOND CHROMOSOME',
                'LEFT GENE', 'RIGHT GENE', 'GENE_PAIR', 'GROUPED TYPE', 'NOTATION'
            ]]
            
            for subgroup_id, group in df_sorted.groupby('subgroup', sort=False):
                # group = group.copy()
                group = group.sort_values(by='translocation_depth', ascending=False, na_position='last').copy()
                group_size = len(group)
                # summary_label = f"{group.iloc[0]['NOTATION']} {group.iloc[0]['LEFT GENE']}::{group.iloc[0]['RIGHT GENE']} {group.iloc[0]['TYPE']}"
                summary_label = f"GROUP: {group.iloc[0]['NOTATION']} {group.iloc[0]['LEFT GENE']}::{group.iloc[0]['RIGHT GENE']}"
                notation = group.iloc[0]['NOTATION']
                left_gene = group.iloc[0]['LEFT GENE']
                right_gene = group.iloc[0]['RIGHT GENE']
                rearr_type = group.iloc[0]['GROUPED TYPE']
                gene_pair = group.iloc[0]['GENE_PAIR']
            
                left_chr = extract_chromosome(group.iloc[0]["LEFT SIDE (5') OF NON-INVERTED BREAKPOINT"])
                right_chr = extract_chromosome(group.iloc[0]["RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT"])
                
                left_range = f"chr{left_chr} {int(group['LEFT COORDINATE'].min())}-{int(group['LEFT COORDINATE'].max())}"
                right_range = f"chr{right_chr} {int(group['RIGHT COORDINATE'].min())}-{int(group['RIGHT COORDINATE'].max())}"
            
                if group_size == 1:
                    ungrouped_blocks.append(group)
        
            # Add ungrouped rows if present
            if ungrouped_blocks:
                ungrouped = pd.concat(ungrouped_blocks, ignore_index=True)
                ungrouped = ungrouped.sort_values(by='translocation_depth', ascending=False, na_position='last')
            
                # Define GENE_PAIRs of interest
                pairs_of_interest = {
                    'MYH11::CBFB', 'CBFB::MYH11',
                    'RUNX1::RUNX1T1', 'RUNX1T1::RUNX1',
                    'DEK::NUP214', 'NUP214::DEK',
                    'KMT2A::MLLT3', 'MLLT3::KMT2A',
                    'BCR::ABL1', 'ABL1::BCR',
                    'PML::RARA', 'RARA::PML'
                }
                ungrouped_not_of_interest = ungrouped[~ungrouped['GENE_PAIR'].isin(pairs_of_interest)].copy() #i.e. not in list of interesting gene pairs
                ungrouped_not_of_interest = ungrouped_not_of_interest.sort_values(
                    by=['GENE_PAIR', 'translocation_depth'], ascending=[True, False]
                )
    
                fixed_columns = ['SAMPLE NAME', 'NOTATION', "LEFT SIDE (5') OF NON-INVERTED BREAKPOINT", "RIGHT SIDE (3') OF NON-INVERTED BREAKPOINT",
                                 'GROUPED TYPE', 'TYPE', 'translocation_depth', 'normal_depth_left', 'normal_depth_right', 'max_normal_depth',
                                 'VAF_left', 'VAF_right', 'max_VAF', 'evidence_count', 'discordant reads', 'split reads']
                other_columns = [col for col in ungrouped_not_of_interest.columns if col not in fixed_columns]
                new_order = fixed_columns + other_columns
                ungrouped_not_of_interest = ungrouped_not_of_interest[new_order]
                ungrouped_not_of_interest = ungrouped_not_of_interest.drop(columns=['row_id', 'subgroup', 'SORT_ORDER', 'subgroup_size'], errors='ignore')
                
                if not ungrouped_not_of_interest.empty:
                    # blank_row = pd.DataFrame([{col: '' for col in group.columns}])
                    combined_ungrouped.append(ungrouped_not_of_interest)
                    # combined_ungrouped.append(blank_row)
                else:
                    print('no ungrouped not of interest events for '+sample)

    if combined_ungrouped:
        df_all_ungrouped = pd.concat(combined_ungrouped, ignore_index=True)
        # Save final_df to Excel with formatting
        output_file = 'UKCTOCS_final_timepoints/Combined_UNGROUPED_NOT_OF_INTEREST_sections_'+cases_or_controls+'_v3.xlsx'

        #remove rearrangements that aren't in the correct orientation
        df_filtered_ungrouped = df_all_ungrouped[df_all_ungrouped.apply(matches_expected_inversion, axis=1)].copy()
        #apply read depth and evidence filters
        df_filtered_ungrouped = df_filtered_ungrouped[df_filtered_ungrouped.apply(passes_read_evidence_filter, axis=1)].copy()
        df_filtered_ungrouped = df_filtered_ungrouped.replace("", pd.NA).dropna(how='all')

        # normalize empties to NA
        df_filtered_ungrouped = df_filtered_ungrouped.applymap(lambda x: x.strip() if isinstance(x, str) else x)
        df_filtered_ungrouped = df_filtered_ungrouped.replace(r'^\s*$', pd.NA, regex=True)
        is_blank = df_filtered_ungrouped.isna().all(axis=1)
        keep = ~is_blank | (is_blank & ~is_blank.shift(fill_value=False))  # keep first of each blank run
        df_filtered_ungrouped = df_filtered_ungrouped[keep]
        # drop leading/trailing blank rows
        if not df_filtered_ungrouped.empty and df_filtered_ungrouped.iloc[0].isna().all():
            df_filtered_ungrouped = df_filtered_ungrouped.iloc[1:]
        if not df_filtered_ungrouped.empty and df_filtered_ungrouped.iloc[-1].isna().all():
            df_filtered_ungrouped = df_filtered_ungrouped.iloc[:-1]
        df_filtered_ungrouped = df_filtered_ungrouped.reset_index(drop=True)
        
        with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
            df_filtered_ungrouped.to_excel(writer, index=False, sheet_name='Summary')
        
            workbook  = writer.book
            worksheet = writer.sheets['Summary']
        
            # Freeze the top row
            worksheet.freeze_panes(1, 0)

            ncols = len(df_filtered_ungrouped.columns)
            
            # Explicitly unhide all columns first (just in case)
            worksheet.set_column(0, ncols - 1, None, None, {'hidden': False})
            
            # Sensible default width so nothing starts at 0
            worksheet.set_column(0, ncols - 1, 10)
        
            # Autosize with a minimum width guard
            for i, col in enumerate(df_filtered_ungrouped.columns):
                series = df_filtered_ungrouped[col].astype(str)
                max_len = max(series.map(len).max(), len(str(col)))
                width = max(8, min(max_len, 35))  # enforce min width of 8 chars
                worksheet.set_column(i, i, width)
                    
            # Narrower widths for P to BC
            for col_idx in range(16, 56):
                worksheet.set_column(col_idx, col_idx, 8)
    
    return print("✅ Done")

In [ ]:
create_combined_ungrouped_not_interesting_spreadsheet('CASES', cases)
create_combined_ungrouped_not_interesting_spreadsheet('CONTROLS', controls)

# Create one overall final dataframe

In [ ]:
def make_overall_spreadsheet(cases_or_controls):

    grouped_path = "UKCTOCS_final_timepoints/Combined_GROUPED_sections_"+cases_or_controls+"_v3.xlsx"
    ungrouped_path = "UKCTOCS_final_timepoints/Combined_UNGROUPED_sections_"+cases_or_controls+"_v3.xlsx"
    not_interesting_path = "UKCTOCS_final_timepoints/Combined_UNGROUPED_NOT_OF_INTEREST_sections_"+cases_or_controls+"_v3.xlsx"
    out_path = "UKCTOCS_final_timepoints/Combined_OVERALL_"+cases_or_controls+".xlsx"
    sheet_name = "Summary"
    
    def load(fp, source_label):
        df = pd.read_excel(fp, sheet_name=sheet_name)
        df["SOURCE"] = source_label
        return df

    df_g = load(grouped_path,   "GROUPED")
    df_u = load(ungrouped_path, "UNGROUPED")
    df_c = load(not_interesting_path,  "NOT INTERESTED")

    # Ensure GROUP TAG exists in all frames
    for d in (df_u, df_c):
        if "GROUP TAG" not in d.columns:
            d["GROUP TAG"] = None

    # Build a unified column order (start from grouped, then append any new cols)
    col_order = list(df_g.columns)
    for d in (df_u, df_c):
        for col in d.columns:
            if col not in col_order:
                col_order.append(col)

    # Reindex to unified columns
    df_g = df_g.reindex(columns=col_order)
    df_u = df_u.reindex(columns=col_order)
    df_c = df_c.reindex(columns=col_order)

    # Concatenate
    overall = pd.concat([df_g, df_u, df_c], ignore_index=True)
    # overall = overall.replace("", pd.NA).dropna(how='all')

    # normalize empties to NA
    overall = overall.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    overall = overall.replace(r'^\s*$', pd.NA, regex=True)
    is_blank = overall.isna().all(axis=1)
    keep = ~is_blank | (is_blank & ~is_blank.shift(fill_value=False))  # keep first of each blank run
    overall = overall[keep]
    # drop leading/trailing blank rows
    if not overall.empty and overall.iloc[0].isna().all():
        overall = overall.iloc[1:]
    if not overall.empty and overall.iloc[-1].isna().all():
        overall = overalld.iloc[:-1]
    overall = overall.reset_index(drop=True)

    # Save nicely formatted
    with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:
        overall.to_excel(writer, index=False, sheet_name=sheet_name)
        ws = writer.sheets[sheet_name]
        ws.freeze_panes(1, 0)
        for i, col in enumerate(overall.columns):
            max_len = max(len(str(col)), overall[col].astype(str).map(len).max())
            ws.set_column(i, i, min(max_len, 35))

    return print("✅ Done")

In [ ]:
# Reciprocal -> canonical mapping (excluding MYH11/CBFB)
RECIP_TO_CANON = {
    'MLLT3::KMT2A':   'KMT2A::MLLT3',
    'ABL1::BCR':      'BCR::ABL1',
    'RARA::PML':      'PML::RARA',
    'RUNX1T1::RUNX1': 'RUNX1::RUNX1T1',
    'NUP214::DEK':    'DEK::NUP214',
}

# Mutual pair for inv(16)
MUTUAL_PAIR = {
    'MYH11::CBFB': 'CBFB::MYH11',
    'CBFB::MYH11': 'MYH11::CBFB'
}

def _norm(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip()

def apply_reciprocal_canonical_rule(overall: pd.DataFrame) -> pd.DataFrame:
    required = {'SAMPLE NAME','GENE_PAIR','translocation_depth','evidence_count','discordant reads'}
    missing = required - set(overall.columns)
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    df = overall.copy()
    df['SAMPLE NAME'] = _norm(df['SAMPLE NAME'])
    df['GENE_PAIR']   = _norm(df['GENE_PAIR'])

    # Spacers (always keep)
    is_spacer = df['GENE_PAIR'].isna() | (df['GENE_PAIR'] == '') | (df['GENE_PAIR'] == 'nan')

    # Thresholds per row
    depth      = pd.to_numeric(df['translocation_depth'], errors='coerce').fillna(0)
    evidence   = pd.to_numeric(df['evidence_count'], errors='coerce').fillna(0)
    discordant = pd.to_numeric(df['discordant reads'], errors='coerce').fillna(0)

    meets_threshold = ((depth < 50)  & (evidence >= 3) & (discordant >= 3)) | \
                      ((depth >= 50) & (evidence >= 5) & (discordant >= 5))
    df['__meets__'] = meets_threshold

    gp     = df['GENE_PAIR']
    sample = df['SAMPLE NAME']

    # Lookup tables
    pass_lookup = df.set_index(['SAMPLE NAME','GENE_PAIR'])['__meets__'].to_dict()
    present_lookup = set(zip(sample, gp))

    # --- Mutual pair rule (MYH11/CBFB) ---
    is_mutual = gp.isin(MUTUAL_PAIR.keys())
    partner   = gp.map(MUTUAL_PAIR)

    partner_present = [(s, p) in present_lookup if isinstance(p, str) else False
                       for s, p in zip(sample, partner)]
    partner_passes  = [pass_lookup.get((s, p), False) if isinstance(p, str) else False
                       for s, p in zip(sample, partner)]

    keep_mutual = [
        (self_pass or partner_pass) if present else self_pass
        for self_pass, partner_pass, present in zip(df['__meets__'], partner_passes, partner_present)
    ]
    keep_mutual = is_mutual & pd.Series(keep_mutual, index=df.index)

    # --- Normal reciprocal rule ---
    is_recip = gp.isin(RECIP_TO_CANON.keys())
    canonical_partner = gp.map(RECIP_TO_CANON)
    canonical_passes_same_sample = pd.Series(
        [pass_lookup.get((s, c), False) if isinstance(c, str) else False
         for s, c in zip(sample, canonical_partner)],
        index=df.index
    )
    keep_recip = is_recip & canonical_passes_same_sample

    # --- Combine ---
    keep = is_spacer | keep_mutual | (~is_recip & ~is_mutual) | keep_recip

    return df[keep].drop(columns='__meets__')

In [ ]:
def make_overall_spreadsheet(cases_or_controls):

    grouped_path = "UKCTOCS_final_timepoints/Combined_GROUPED_sections_"+cases_or_controls+"_v3.xlsx"
    ungrouped_path = "UKCTOCS_final_timepoints/Combined_UNGROUPED_sections_"+cases_or_controls+"_v3.xlsx"
    not_interesting_path = "UKCTOCS_final_timepoints/Combined_UNGROUPED_NOT_OF_INTEREST_sections_"+cases_or_controls+"_v3.xlsx"
    out_path = "UKCTOCS_final_timepoints/Combined_OVERALL_"+cases_or_controls+".xlsx"
    sheet_name = "Summary"
    
    def load(fp, source_label):
        df = pd.read_excel(fp, sheet_name=sheet_name)
        df["SOURCE"] = source_label
        return df

    df_g = load(grouped_path,   "GROUPED")
    df_u = load(ungrouped_path, "UNGROUPED")
    df_c = load(not_interesting_path,  "NOT INTERESTED")

    # Ensure GROUP TAG exists in all frames
    for d in (df_u, df_c):
        if "GROUP TAG" not in d.columns:
            d["GROUP TAG"] = None

    # Build a unified column order (start from grouped, then append any new cols)
    col_order = list(df_g.columns)
    for d in (df_u, df_c):
        for col in d.columns:
            if col not in col_order:
                col_order.append(col)

    # Reindex to unified columns
    df_g = df_g.reindex(columns=col_order)
    df_u = df_u.reindex(columns=col_order)
    df_c = df_c.reindex(columns=col_order)

    # Concatenate
    overall = pd.concat([df_g, df_u, df_c], ignore_index=True)
    overall = apply_reciprocal_canonical_rule(overall)

    # --- Clean rows with missing sample name unless they are true spacers ---
    overall = overall.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    overall = overall.replace(r'^\s*$', pd.NA, regex=True).replace({"nan": pd.NA, "None": pd.NA})

    is_spacer = overall['GENE_PAIR'].isna() | (overall['GENE_PAIR'] == "")
    overall = overall[~(overall['SAMPLE NAME'].isna() & ~is_spacer)]

    # --- Treat rows as "blank" ignoring admin columns like SOURCE (and GROUP TAG) ---
    non_data_cols = ['SOURCE', 'GROUP TAG']  # add any other non-data columns here
    data_cols = [c for c in overall.columns if c not in non_data_cols]
    if not data_cols:
        data_cols = overall.columns  # fallback (unlikely)

    is_blank = overall[data_cols].isna().all(axis=1)

    # Keep only the first of any consecutive blank rows
    keep = ~is_blank | (is_blank & ~is_blank.shift(fill_value=False))
    overall = overall[keep]

    # Drop leading/trailing blank rows (based on data columns)
    while not overall.empty and overall.iloc[0][data_cols].isna().all():
        overall = overall.iloc[1:]
    while not overall.empty and overall.iloc[-1][data_cols].isna().all():
        overall = overall.iloc[:-1]

    overall = overall.reset_index(drop=True)

    ## Add information to blank GROUP TAG columns
    mask = (
        (overall['GROUP TAG'].isna() | (overall['GROUP TAG'].astype(str).str.strip() == ''))
        & (~overall['GENE_PAIR'].isna()) 
        & (overall['GENE_PAIR'].astype(str).str.strip() != '')
    )
    label = (
        'UNGROUPED: '
        + overall['NOTATION'].astype('string').fillna('').str.strip()
        + ' '
        + overall['GENE_PAIR'].astype('string').fillna('').str.strip()
    ).str.replace(r'\s+', ' ', regex=True).str.strip()
    overall.loc[mask, 'GROUP TAG'] = label
    
    if not overall.empty:
        grouped_frames = []
        for sample, group in overall.groupby("SAMPLE NAME", sort=True):
            group_sorted = group.sort_values("translocation_depth", ascending=False, na_position="last")
            grouped_frames.append(group_sorted)
            # add a blank row between sample groups
            blank_row = pd.DataFrame([{col: pd.NA for col in overall.columns}])
            grouped_frames.append(blank_row)
        # remove last blank row
        if grouped_frames:
            grouped_frames = grouped_frames[:-1]
        overall = pd.concat(grouped_frames, ignore_index=True)

    # Save nicely formatted
    with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:
        overall.to_excel(writer, index=False, sheet_name=sheet_name)
        ws = writer.sheets[sheet_name]
        ws.freeze_panes(1, 0)
        for i, col in enumerate(overall.columns):
            max_len = max(len(str(col)), overall[col].astype(str).map(len).max())
            ws.set_column(i, i, min(max_len, 35))

    return print("✅ Done")

In [ ]:
make_overall_spreadsheet('CASES')
make_overall_spreadsheet('CONTROLS')

In [ ]:
def make_overall_spreadsheet(positive_controls):

    grouped_path = "Combined_GROUPED_sections_POSITIVE_CONTROLS_v3.xlsx"
    ungrouped_path = "Combined_UNGROUPED_sections_POSITIVE_CONTROLS_v3.xlsx"
    not_interesting_path = "Combined_UNGROUPED_NOT_OF_INTEREST_sections_POSITIVE_CONTROLS_v3.xlsx"
    out_path = "Combined_OVERALL_POSITIVE_CONTROLS.xlsx"
    sheet_name = "Summary"
    
    def load(fp, source_label):
        df = pd.read_excel(fp, sheet_name=sheet_name)
        df["SOURCE"] = source_label
        return df

    df_g = load(grouped_path,   "GROUPED")
    df_u = load(ungrouped_path, "UNGROUPED")
    df_c = load(not_interesting_path,  "NOT INTERESTED")

    # Ensure GROUP TAG exists in all frames
    for d in (df_u, df_c):
        if "GROUP TAG" not in d.columns:
            d["GROUP TAG"] = None

    # Build a unified column order (start from grouped, then append any new cols)
    col_order = list(df_g.columns)
    for d in (df_u, df_c):
        for col in d.columns:
            if col not in col_order:
                col_order.append(col)

    # Reindex to unified columns
    df_g = df_g.reindex(columns=col_order)
    df_u = df_u.reindex(columns=col_order)
    df_c = df_c.reindex(columns=col_order)

    # Concatenate
    overall = pd.concat([df_g, df_u, df_c], ignore_index=True)
    overall = apply_reciprocal_canonical_rule(overall)

    # --- Clean rows with missing sample name unless they are true spacers ---
    overall = overall.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    overall = overall.replace(r'^\s*$', pd.NA, regex=True).replace({"nan": pd.NA, "None": pd.NA})

    is_spacer = overall['GENE_PAIR'].isna() | (overall['GENE_PAIR'] == "")
    overall = overall[~(overall['SAMPLE NAME'].isna() & ~is_spacer)]

    # --- Treat rows as "blank" ignoring admin columns like SOURCE (and GROUP TAG) ---
    non_data_cols = ['SOURCE', 'GROUP TAG']  # add any other non-data columns here
    data_cols = [c for c in overall.columns if c not in non_data_cols]
    if not data_cols:
        data_cols = overall.columns  # fallback (unlikely)

    is_blank = overall[data_cols].isna().all(axis=1)

    # Keep only the first of any consecutive blank rows
    keep = ~is_blank | (is_blank & ~is_blank.shift(fill_value=False))
    overall = overall[keep]

    # Drop leading/trailing blank rows (based on data columns)
    while not overall.empty and overall.iloc[0][data_cols].isna().all():
        overall = overall.iloc[1:]
    while not overall.empty and overall.iloc[-1][data_cols].isna().all():
        overall = overall.iloc[:-1]

    overall = overall.reset_index(drop=True)

    ## Add information to blank GROUP TAG columns
    mask = (
        (overall['GROUP TAG'].isna() | (overall['GROUP TAG'].astype(str).str.strip() == ''))
        & (~overall['GENE_PAIR'].isna()) 
        & (overall['GENE_PAIR'].astype(str).str.strip() != '')
    )
    label = (
        'UNGROUPED: '
        + overall['NOTATION'].astype('string').fillna('').str.strip()
        + ' '
        + overall['GENE_PAIR'].astype('string').fillna('').str.strip()
    ).str.replace(r'\s+', ' ', regex=True).str.strip()
    overall.loc[mask, 'GROUP TAG'] = label
    
    if not overall.empty:
        grouped_frames = []
        for sample, group in overall.groupby("SAMPLE NAME", sort=True):
            group_sorted = group.sort_values("translocation_depth", ascending=False, na_position="last")
            grouped_frames.append(group_sorted)
            # add a blank row between sample groups
            blank_row = pd.DataFrame([{col: pd.NA for col in overall.columns}])
            grouped_frames.append(blank_row)
        # remove last blank row
        if grouped_frames:
            grouped_frames = grouped_frames[:-1]
        overall = pd.concat(grouped_frames, ignore_index=True)

    # Save nicely formatted
    with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:
        overall.to_excel(writer, index=False, sheet_name=sheet_name)
        ws = writer.sheets[sheet_name]
        ws.freeze_panes(1, 0)
        for i, col in enumerate(overall.columns):
            max_len = max(len(str(col)), overall[col].astype(str).map(len).max())
            ws.set_column(i, i, min(max_len, 35))

    return print("✅ Done")

In [ ]:
make_overall_spreadsheet('POSITIVE CONTROLS')